# openpilot's driving model on comma2k19, open loop: what the recorded predictions say

> **Outputs are not committed: run the notebook to see them.** Every figure, table and number the text refers to comes from a run, and git keeps none of them: `tests/test_notebooks_are_bare.py` holds this file to that.
>
> 1. **Once:** `make setup`, then `uv pip install --python .venv/bin/python -e ".[analysis]"` for Jupyter and what the notebook draws with.
> 2. **The site:** `paddock.toml` (or `PADDOCK_CONFIG`) says where `{runs}` and the data are. It reads a comma2k19 openpilot run's records under `{runs}` (`RUN_ID`, §1; `None` takes the newest valid run), about 100 MB of `samples.jsonl`; where the site has it, scenario mining's table, `{repos}/scenario_mining/comma2k19/data/scenario_metadata.csv` (without it there are no `ctx_` columns, and `"pilot"` becomes `"random"`); and the comma2k19 segments for the frames §13 draws.
> 3. **Run it:** run all on the repo's `.venv` kernel, or headless from the repo root: `.venv/bin/jupyter execute --output='{notebook_name}_all_executed' tools/models/openpilot/analysis/openpilot_drv_supcombo_comma2k19_openloop_analysis.ipynb`. That writes an executed copy beside this file, which git ignores. Never `--inplace`: it writes the outputs into this file, and `make check` fails on that. A run in Jupyter or VS Code saves them here too; strip it before committing, with `.venv/bin/python -m tools.strip_notebooks tools/models/openpilot/analysis/openpilot_drv_supcombo_comma2k19_openloop_analysis.ipynb`.
>
> **What it needs.** No model, no node, no GPU. `SUBSET` (§1) is 500 keyframes by default and `"all"` is every keyframe of the run. Where it writes is under **Outputs** below.

This notebook reads one paddock run's per-keyframe records, tags every keyframe with simple threshold rules, groups the tags into scenario clusters, failure modes and strengths, and draws representative keyframes. It never runs the model.

**What was run.** openpilot's driving model, as paddock's openpilot adapter runs it: **model inference only**. calibrationd, plannerd and controlsd do not run. In their place paddock supplies a route-pooled calibration (calibrationd's formula on the recorded pose, `run.json` → `config.calibration`) and **no desire or navigation input**, so the model is never told the route.

**The prediction** is the plan head's positions resampled to 20 waypoints every 0.5 s (10 s), in openpilot's calibrated (car-aligned) frame: x forward, y left, metres. The other heads are recorded in each sample's `extras`: pose (speed, yaw rate), the plan's σ, the lead, lane lines and the road.

**Which weights.** This run's `run.json` records no provenance. From the openpilot checkout's history it was v0.11.1's split graphs (`driving_vision` + `driving_policy`), not the pinned `driving_supercombo.onnx` that the kitscenes run used: the checkout sat at `4df40d2c1` from 10 to 18 September, and the run started on the 16th. On the five turns the [comma2k19 image-tweak notebook](openpilot_drv_supcombo_comma2k19_image_tweak.ipynb) checks, the pinned model reproduces this run's plans to within about 2 cm over 3 s, so there the two plan alike; whether they do on every keyframe is the re-sweep at the pin the ROADMAP lists.

**The GT frame (§4).** comma2k19 scores against the raw device-pose path, which carries the phone's mount yaw (1–2°). The plan comes out in the calibrated frame, which does not. Rotating the GT by the stored calibration yaw (`rpy_calib_deg[2]`) takes FDE@3s over all 10,165 keyframes from a mean of 2.71 m to 1.64 m (median 2.80 → 1.27 m). `GT_FRAME` in §1 picks the GT every comparison uses; §4 re-checks the finding on whatever is loaded, and §7 shows both frames side by side.

## How to use it

- **Run all.** §1 is the only cell to edit for a normal run. Headless, on every keyframe: `env PADDOCK_ANALYSIS_SUBSET=all .venv/bin/jupyter execute <this notebook>`, which leaves this file untouched and writes §14's export. To keep the executed copy too, add `--output='{notebook_name}_all_executed'`: it lands beside this notebook, and git ignores `*_executed.ipynb`. Keep dots out of that name: `jupyter execute` replaces everything after the last dot with `.ipynb`, so `{notebook_name}.all` would overwrite this notebook.
- **Which keyframes.** `SUBSET = "pilot"` is the 5% of segments in the scenario-mining CSV (100 segments, 500 keyframes), which also brings the CSV's context tags; a site without scenario mining's checkout gets `"random"` instead, and no `ctx_` columns. `"random"` draws `FRACTION` of the segments with `SEED`. `"all"` is every keyframe of the run.
- **Debug one thing.** Set `FOCUS` to one segment (`"<route>/<segment>/"`, keep the trailing slash), one route (`"<route>/"`) or a list of exact sample ids. Every section then runs on just those keyframes, §13 draws each of them, and `explain(sample_id)` (§6) says why every tag fired.
- **Try other values.** Every helper takes keyword arguments that default to §1 and to `P`, so the cell below any output can call it again with other values. Every helper that works on a set of keyframes takes a condition the same way: `gt_lat="curve_R"`, `speed_band=["mid", "high"]`, `fm_understeer=True`, plus an optional `where="<pandas query>"` such as `where="speed_ratio < 0.9 and ref_speed > @v_mid"` (`@name` is a param).
- **Point and click.** `score_explorer()` (§7), `explore()` (§11) and `browse()` (§13) wrap the same functions in widgets. `explore()` prints the `COND` and `WHERE` it applied, so a selection made with the mouse can be pasted into a cell and kept. The widgets are off until you set `WIDGETS = True` in §1, which says why.
- **Change a tag.** Every threshold and rule is in `openpilot_drv_supcombo_tags.yaml` beside this notebook, shared with the kitscenes notebook. To change one for this run only: `OVERRIDES` in §1 (params), `retag(params=P | {...})` for a side-by-side comparison, or a modified copy of `CFG` (§6). Then re-run from §6.

**Outputs.** Only `export()` (§14) writes tables: to `{runs}/comma2k19/_analysis/openpilot/<run_id>/<subset>/`, with a manifest. Decoded keyframes are cached as JPEG under `{runs}/comma2k19/_analysis/_frames/`. Nothing is written under `results/`.

**The outputs are not committed** (top of this notebook): this file is the code and the prose, and a run is what draws the figures. Leave `WIDGETS` off for a headless run, as it is by default: a saved widget cannot be drawn once its kernel is gone (§1). An executed copy is made to be shared, so nothing a cell prints may name a machine path: print paths relative to `{runs}`, as §14 does. Keyframe figures are drawn as JPEG (`_show_photo`, §13): their camera panels are most of an executed copy's size.

| § | section |
|---|---|
| 1 | Parameters |
| 2 | Tag config and glossary |
| 3 | Load |
| 4 | Frame check: recorded vs calibrated GT |
| 5 | Features |
| 6 | Tags: prevalence, `select()`, `explain()` |
| 7 | Scores: recorded, recomputed, `scores()`, `score_explorer()`, FDE vs horizon, yaw sweep |
| 8 | Distributions, with the thresholds drawn in |
| 9 | Scenario clusters |
| 10 | Confusion: GT class vs plan class |
| 11 | Conditions: `profile()`, `explore()` |
| 12 | Failure modes and strengths |
| 13 | Keyframes: `show()`, `sheet()`, `control()`, `browse()` |
| 14 | Export |
| 15 | Scratch |

In [ ]:
import base64
import contextlib
import hashlib
import html
import io
import itertools
import json
import os
import re
import textwrap
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython import get_ipython
from IPython.core.displaypub import DisplayPublisher
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

from paddock.adapters.benchmarks.comma2k19 import loader as L
from paddock.adapters.benchmarks.comma2k19 import overlay as OV
from paddock.adapters.benchmarks.comma2k19.metrics import score_trajectory
from paddock.config import REPO_ROOT, SiteConfig
from paddock.metrics.comfort import BOUNDS, comfort
from paddock.render import load_runs
from paddock.results.store import config_hash
from paddock.viz import bev, palette

try:
    import ipywidgets as W
except ImportError:  # the widgets are conveniences; each one wraps a function you can call directly
    W = None

# retag() adds tag columns one at a time, for readability, and defragments once at the end
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

## 1 Parameters

The only cell to edit for a normal run. `SUBSET`, `FOCUS` and `WIDGETS` can also come from the environment (`PADDOCK_ANALYSIS_SUBSET`, `PADDOCK_ANALYSIS_FOCUS`, `PADDOCK_ANALYSIS_WIDGETS=1`), which is how a headless run or a session sets them without editing this cell.

- `FOCUS` overrides `SUBSET`. A string is a prefix of the sample id after its leading `comma2k19/<chunk>/`, so `"<route>/<segment>/"` is one segment's 5 keyframes and `"<route>/"` is a whole route. A list is exact sample ids.
- `OVERRIDES` replaces tag params for this run only. An unknown name raises, so a typo cannot silently change nothing.
- `GT_FRAME = "calibrated"` compares the plan with the GT rotated into the plan's frame; `"recorded"` with the GT the run was scored against (§4).
- `WIDGETS = True` gives `score_explorer()` (§7), `explore()` (§11) and `browse()` (§13) their controls; off, each prints what to call instead. Off is the default because a saved widget is only the id of an object in the kernel that drew it: reopened later, VS Code shows `Could not render content for 'application/vnd.jupyter.widget-view+json'` in its place. Leave it off for a headless run (§0).

In [ ]:
BENCH, MODEL = "comma2k19", "openpilot"
RUN_ID = None  # None = the newest valid run (printed in §3)
SUBSET = os.environ.get("PADDOCK_ANALYSIS_SUBSET", "pilot")  # "pilot" (the 5% CSV) | "random" | "all"
# "random" draws this fraction of whole segments; SEED also picks the keyframes shown
FRACTION, SEED = 0.05, 0
FOCUS = os.environ.get("PADDOCK_ANALYSIS_FOCUS") or None  # debug one thing; overrides SUBSET
# FOCUS = "b0c9d2329ad1606b|2018-07-27--06-03-57/3/"  # one segment: its 5 keyframes (keep the trailing /)
# FOCUS = "b0c9d2329ad1606b|2018-07-27--06-03-57/"    # one route
# FOCUS = ["comma2k19/1/b0c9d2329ad1606b|2018-07-27--06-03-57/3/159"]  # exact sample ids
OVERRIDES = {}  # tag params for this run only, e.g. {"a_brake": -0.7, "horizon_s": 2.0}
GT_FRAME = "calibrated"  # or "recorded": the GT the run was scored against (§4)
RENDER, MAX_RENDER = True, 40  # §13: draw keyframes at all; at most this many per call
WIDGETS = os.environ.get("PADDOCK_ANALYSIS_WIDGETS") == "1"  # True: §7, §11 and §13 draw controls
TAGS = REPO_ROOT / "tools/models/openpilot/analysis/openpilot_drv_supcombo_tags.yaml"

## 2 Tag config and glossary

Loads the shared tags file into `CFG`. `P` is its `params`, then this benchmark's `bench_params` (none on comma2k19), then `OVERRIDES`; every helper reads thresholds from `P` unless given `params=`. A rule marked `only: [kitscenes]` is skipped here, and the list of skipped rules is printed.

The glossary below is generated from the file itself, so it cannot drift from what runs: every param with its value and doc, then every class (rules in order, first match wins), bin, failure mode and named strength. `export()` saves it as `glossary.md`. The feature columns the rules refer to are documented in §5.

In [ ]:
cfg = SiteConfig.load()
TAG_TEXT = TAGS.read_text()
CFG = yaml.safe_load(TAG_TEXT)
BENCH_PARAMS = CFG.get("bench_params", {}).get(BENCH, {})
unknown = sorted((set(OVERRIDES) | set(BENCH_PARAMS)) - set(CFG["params"]))
if unknown:
    raise KeyError(f"OVERRIDES or bench_params.{BENCH} name no param in {TAGS.name}: {unknown}")
BASE = CFG["params"] | BENCH_PARAMS  # the tags file's values for this benchmark
P = BASE | OVERRIDES
for k, v in BENCH_PARAMS.items():
    print(f"{BENCH}: {k} = {v} (params: {CFG['params'][k]})")
for k, v in OVERRIDES.items():
    print(f"override {k}: {BASE[k]} -> {v}")


def applies(rule, bench=BENCH):
    """Whether a class, bin, mode or strength is meant for this benchmark (its `only:` list)."""
    return bench in rule.get("only", [bench])


def param_docs(text=None):
    """{param: (section, doc)}, read from the comments of the tags file's params block."""
    lines = (text or TAG_TEXT).splitlines()
    docs, section, last = {}, "", None
    for line in lines[lines.index("params:") + 1 :]:
        if re.match(r"[A-Za-z]", line):
            break
        if m := re.match(r"  # -- (.*) --", line):
            section = m.group(1)
        elif m := re.match(r"  (\w+):[^#]*(?:#\s*(.*))?$", line):
            last = m.group(1)
            docs[last] = (section, m.group(2) or "")
        elif (m := re.match(r" {3,}#\s*(.*)", line)) and last:
            docs[last] = (docs[last][0], f"{docs[last][1]} {m.group(1)}")
    return docs


def _outputs(name, spec):
    """The columns a class writes: gt_<name> and pred_<name> for a sided class, else <name>."""
    return [f"{s}_{name}" for s in spec["sides"]] if "sides" in spec else [name]


def _md(text):
    """Text that is safe inside a markdown table cell."""
    return str(text).replace("|", "\\|").replace("\n", " ")


def glossary(params=None, tags=None, bench=BENCH):
    """Every param, class, bin, failure mode and named strength in the tags file, with its doc, as markdown."""
    p, tags = params or P, tags or CFG
    docs = param_docs()
    own = tags.get("bench_params", {}).get(bench, {})
    out = ["### Parameters", "", "| section | param | value | doc |", "|---|---|---|---|"]
    for k, v in p.items():
        section, doc = docs.get(k, ("", ""))
        base = own.get(k, tags["params"].get(k))
        value = f"**{v}** (yaml: {base})" if v != base else f"{v}"
        if k in own:
            value += f" ({bench}; params: {tags['params'][k]})"
        out.append(f"| {section} | `{k}` | {value} | {_md(doc)} |")
    out += ["", "### Classes", "", "Rules are tried in order and the first that matches wins.", ""]
    for name, spec in tags["classes"].items():
        if not applies(spec, bench):
            continue
        cols = ", ".join(f"`{c}`" for c in _outputs(name, spec))
        out.append(f"- {cols} ({spec['group']}): {spec['doc']}")
        if spec.get("requires"):
            out.append(
                f"    - `{spec.get('missing', 'na')}` where any of {', '.join(spec['requires'])} is NaN"
            )
        for label, rule in spec["rules"].items():
            out.append(f"    - `{label}` if `{rule['when']}`: {rule.get('doc', '')}")
        default_doc = f": {spec['default_doc']}" if "default_doc" in spec else ""
        out.append(f"    - otherwise `{spec['default']}`{default_doc}")
    out += ["", "### Bins (right-open intervals)", ""]
    for name, spec in tags["bins"].items():
        if applies(spec, bench):
            edges = ", ".join(
                f"{p[e[1:]]} ({e[1:]})" if isinstance(e, str) else f"{e}" for e in spec["edges"]
            )
            out.append(
                f"- `{name}` ({spec['group']}): `{spec['col']}` cut at {edges} into "
                f"{', '.join(spec['labels'])}; NaN is `{spec.get('missing', 'na')}`. {spec['doc']}"
            )
    for kind, title in (
        ("failure_modes", "Failure modes: `fm_<name>`, with severity `sev_<name>` (bigger is worse)"),
        ("strengths", "Named strengths: `st_<name>`"),
    ):
        out += ["", f"### {title}", "", "| name | group | when | severity | doc |", "|---|---|---|---|---|"]
        for name, spec in tags[kind].items():
            if applies(spec, bench):
                out.append(
                    f"| {name} | {spec.get('group', '')} | `{_md(spec['when'])}` | "
                    f"`{_md(spec.get('severity', ''))}` | {_md(spec['doc'])} |"
                )
    return "\n".join(out)


skipped = [
    f"{kind}.{name}"
    for kind in ("classes", "bins", "failure_modes", "strengths")
    for name, spec in CFG[kind].items()
    if not applies(spec)
]
print(f"skipped on {BENCH} (only: another benchmark): {', '.join(skipped) or 'none'}")
display(Markdown(glossary()))

## 3 Load

`load()` reads the run's records (once per kernel: `samples.jsonl` is about 100 MB for all of comma2k19) and builds:

- `D`: one row per keyframe. Identity (`vehicle`, `scene` = chunk/route/segment, `keyframe`), the calibration used (`pitch_cal_deg`, `yaw_cal_deg`), the reference signals (`ref_*`: CAN speed and yaw rate, the radar lead), the model's own heads (`op_*`: lead, lane-line probability, pose speed and yaw rate), every recorded metric (`rec_*`, with `@` written `_`), and the pilot CSV's context (`ctx_*`).
- Arrays aligned with `D`'s rows: `PRED` and `GT_RAW` (n × 20 × 2, ego frame), `SIGMA` (n × 20 × 2, the plan's σx, σy), `RPY` (n × 3, degrees).

A row's label in `D` (and in `T`, §6) is its position in these arrays, so do not reset the index of a frame you pass back in as `df=`. The loader raises, naming the run's site, if the records are not on this machine, instead of returning nothing.

In [ ]:
_RUNS = {}


def _run(run_id=RUN_ID):
    """The run and its records, loaded once per kernel."""
    if run_id not in _RUNS:
        runs = load_runs(cfg, BENCH, models=[MODEL], run_ids=[run_id] if run_id else None)
        if not runs:
            raise LookupError(
                f"no {'run ' + run_id if run_id else 'valid run'} of {MODEL} under results/{BENCH}/{MODEL}/"
            )
        run = runs[0]
        if not run.samples:
            raise FileNotFoundError(
                f"{run.run_id} has no samples.jsonl under {{runs}} on this machine; "
                f"it was run on site {run.run.get('site')!r}"
            )
        _RUNS[run_id] = run
    return _RUNS[run_id]


def context_table(bench=BENCH, tags=None):
    """Per-segment metadata from the tags file's `context:` entry, columns prefixed ctx_ (empty if none)."""
    spec = (tags or CFG).get("context", {}).get(bench, {})
    if "csv" not in spec:
        return pd.DataFrame()
    path = cfg.resolve(spec["csv"])
    # Scenario mining is this site's extra, not paddock's: no repos.yaml entry pins it. Without its CSV there
    # are no ctx_ columns, rather than the load failing.
    if not os.path.exists(path):
        return pd.DataFrame()
    table = pd.read_csv(path).set_index(spec["key"])
    keep = [c for c in table.columns if any(re.fullmatch(p.replace("*", ".*"), c) for p in spec["columns"])]
    return table[keep].add_prefix("ctx_")


# "pilot" is the scenario-mining CSV's 5% of segments. A site without that checkout analyses the same share
# drawn at random instead, under the random subset's own name, so the export and its manifest say which it was.
if SUBSET == "pilot" and context_table().empty:
    SUBSET = "random"
    print(f"SUBSET: no scenario-mining CSV ({CFG['context'][BENCH]['csv']}), so 'random': {FRACTION:g} of the segments")


def _scene(s):
    return "{chunk}/{route_id}/{segment}".format(**s["native_id"])


def select_ids(run, subset=SUBSET, focus=FOCUS, fraction=FRACTION, seed=SEED):
    """The sample ids to analyse: FOCUS if set, else SUBSET."""
    ids = [s["sample_id"] for s in run.samples]
    if isinstance(focus, str):
        # ids are comma2k19/<chunk>/<route>/<segment>/<keyframe>; match after dropping 0, 1 or 2 leading parts
        hit = [i for i in ids if any(i.split("/", k)[-1].startswith(focus) for k in range(3))]
    elif focus:
        wanted = set(focus)
        hit = [i for i in ids if i in wanted]
    elif subset == "all":
        return ids
    elif subset == "pilot":
        scenes = set(context_table().index)
        return [s["sample_id"] for s in run.samples if _scene(s) in scenes]
    elif subset == "random":
        scenes = sorted({_scene(s) for s in run.samples})
        size = max(1, round(fraction * len(scenes)))
        pick = set(np.random.default_rng(seed).choice(scenes, size=size, replace=False))
        return [s["sample_id"] for s in run.samples if _scene(s) in pick]
    else:
        raise ValueError(f"SUBSET is 'pilot', 'random' or 'all', not {subset!r}")
    if not hit:
        raise LookupError(f"FOCUS {focus!r} matches no sample id; they look like {ids[0]!r}")
    return hit


N_WP = 20  # waypoints in the plan and the GT: 10 s at 0.5 s


def _pad(rows, n=N_WP):
    """(n, 2) floats, NaN past the end: a GT near the end of a segment can be shorter than 10 s."""
    out = np.full((n, 2), np.nan)
    rows = np.asarray(rows, dtype=float).reshape(-1, 2)[:n]
    out[: len(rows)] = rows
    return out


def _num(v):
    return float("nan") if v is None else float(v)


def load(run_id=RUN_ID, subset=SUBSET, focus=FOCUS, fraction=FRACTION, seed=SEED):
    """(run, D, PRED, GT_RAW, SIGMA, RPY): one row per keyframe, and arrays aligned with the rows."""
    run = _run(run_id)
    keep = set(select_ids(run, subset, focus, fraction, seed))
    samples = [s for s in run.samples if s["sample_id"] in keep]
    rows = []
    for s in samples:
        nid, ex = s["native_id"], s.get("extras") or {}
        ref = nid.get("reference") or {}
        lead = (ex.get("lead") or [{}])[0] or {}
        probs = (ex.get("lane") or {}).get("line_probs") or []
        pose = ex.get("pose") or {}
        rows.append(
            {
                "sample_id": s["sample_id"],
                "vehicle": s["split"],
                "scene": _scene(s),
                "route": nid["route_id"],
                "chunk": nid["chunk"],
                "segment": nid["segment"],
                "keyframe": nid["keyframe_index"],
                "pitch_cal_deg": nid["rpy_calib_deg"][1],
                "yaw_cal_deg": nid["rpy_calib_deg"][2],
                **{
                    f"ref_{k}": _num(ref.get(k))
                    for k in ("speed", "yaw_rate", "lead_d", "lead_y", "lead_v_rel", "n_radar_tracks")
                },
                **{f"op_lead_{k}": _num(lead.get(k)) for k in ("prob", "d", "y", "v_rel")},
                # the two ego lane lines, as auxiliary.py's lane_prob
                "op_lane_prob": float(np.mean(probs[1:3])) if len(probs) >= 3 else np.nan,
                "op_v": _num((pose.get("velocity") or [None])[0]),
                "op_yaw_rate": _num((pose.get("rotation_rate") or [None] * 3)[2]),
                **{f"rec_{k.replace('@', '_')}": _num(v) for k, v in s["metrics"].items()},
            }
        )
    D = pd.DataFrame(rows)
    D["ref_has_lead"] = D["ref_lead_d"].notna()
    # The tags file names these ego_* so each benchmark can supply its own reference (CAN here)
    D["ego_speed_rel_err"] = D["rec_speed_rel_err"]
    D["ego_yaw_bias"] = D["rec_yaw_rate_bias"]
    ctx = context_table()
    if len(ctx):
        D = D.join(ctx, on="scene")
    else:
        print("context: no scenario-mining CSV (tags file, context:), so no ctx_ columns")
    PRED = np.stack([_pad(s["prediction"]) for s in samples])
    GT_RAW = np.stack([_pad(s["ground_truth"]) for s in samples])
    SIGMA = np.stack([_pad((s.get("extras") or {}).get("sigma") or []) for s in samples])
    RPY = np.array([s["native_id"]["rpy_calib_deg"] for s in samples], dtype=float)
    return run, D, PRED, GT_RAW, SIGMA, RPY


RUN, D, PRED, GT_RAW, SIGMA, RPY = load()
YAW_CAL = np.radians(RPY[:, 2])
DT, EVAL_H = RUN.run["config"]["interval_s"], RUN.run["config"]["eval_horizon_s"]
M = RUN.metrics
print(
    f"run {RUN.run_id}  site {RUN.run.get('site')}  started {RUN.run.get('started_utc')}  valid {RUN.run.get('valid')}"
)
prov = RUN.run.get("provenance")
print(
    "provenance:",
    json.dumps(prov, default=str) if prov else "none recorded (see §0 on which weights these are)",
)
chosen = f"focus {FOCUS!r}" if FOCUS else f"subset {SUBSET!r}"
print(
    f"{len(D)} of {len(RUN.samples)} keyframes ({chosen}): {D.scene.nunique()} segments, {D.route.nunique()} routes"
)
print(
    f"GT shorter than 10 s: {int(np.isnan(GT_RAW[:, -1, 0]).sum())} keyframes (their late horizons are NaN, never scored)"
)
display(
    D.groupby("vehicle").agg(
        keyframes=("sample_id", "size"), segments=("scene", "nunique"), routes=("route", "nunique")
    )
)

## 4 Frame check: recorded vs calibrated GT

The run's GT is the raw device-pose path; the plan is in openpilot's calibrated frame. The two differ by the phone's mount rotation, whose yaw is `rpy_calib_deg[2]`, the calibration the adapter itself used. `gt_xy("calibrated")` rotates the GT by that yaw (counter-clockwise, about the ego origin).

This section checks three things on whatever is loaded:

1. **The records are read correctly.** Recomputed against the recorded GT, ADE, FDE@1/3/10 s and cross-track MAE equal the stored per-keyframe metrics (asserted), and paddock's own `score_trajectory` agrees on a sample of keyframes.
2. **The offset is the mount yaw.** On long straight stretches (GT ≥ 100 m, within 0.5 m of its chord), the bearing of the 10 s point should be about the negative mount yaw for the recorded GT, and about 0° for the plan and the calibrated GT.
3. **What it costs.** FDE@3s per vehicle in both frames.

In [ ]:
def rot(xy, yaw):
    """Rotate (k, n, 2) points counter-clockwise about the origin by a per-row yaw (radians)."""
    c, s = np.cos(yaw)[:, None], np.sin(yaw)[:, None]
    return np.stack([c * xy[..., 0] - s * xy[..., 1], s * xy[..., 0] + c * xy[..., 1]], axis=-1)


def gt_xy(frame=None, yaw_offset_deg=0.0):
    """The GT (k, 20, 2) as recorded, or rotated into the plan's calibrated frame; plus an optional extra yaw."""
    frame = frame or GT_FRAME
    if frame not in ("calibrated", "recorded"):
        raise ValueError(f"GT frame is 'calibrated' or 'recorded', not {frame!r}")
    yaw = (YAW_CAL if frame == "calibrated" else np.zeros(len(GT_RAW))) + np.radians(yaw_offset_deg)
    return rot(GT_RAW, yaw)


def split(pred, gt, n):
    """(along, cross) error over the first n waypoints against the GT's own tangent, per keyframe.

    paddock.metrics.openloop.path_tangent_split, vectorised over keyframes: the tangent is np.gradient
    of the GT's first n points (one-sided at the ends). along > 0 is ahead of the GT, cross > 0 left of it.
    """
    p, g = pred[:, :n], gt[:, :n]
    d = np.gradient(g, axis=1) if n > 1 else np.broadcast_to([1.0, 0.0], g.shape)
    norm = np.linalg.norm(d, axis=-1, keepdims=True)
    with np.errstate(invalid="ignore"):
        t = np.where(norm > 1e-6, d / np.maximum(norm, 1e-12), [1.0, 0.0])
    e = p - g
    return (e * t).sum(-1), e[..., 0] * -t[..., 1] + e[..., 1] * t[..., 0]


def disp(gt, n, pred=None):
    """(ADE, FDE) over the first n waypoints, per keyframe; NaN where the GT is shorter."""
    err = np.linalg.norm((PRED if pred is None else pred)[:, :n] - gt[:, :n], axis=-1)
    return err.mean(1), err[:, n - 1]

In [ ]:
n3 = round(EVAL_H / DT)
checks = {
    "rec_ade": disp(GT_RAW, n3)[0],
    "rec_fde_1s": disp(GT_RAW, round(1 / DT))[1],
    "rec_fde_3s": disp(GT_RAW, n3)[1],
    "rec_ade_10s": disp(GT_RAW, N_WP)[0],
    "rec_fde_10s": disp(GT_RAW, N_WP)[1],
    "rec_cross_track_mae": np.abs(split(PRED, GT_RAW, n3)[1]).mean(1),
}
for col, mine in checks.items():
    rec = D[col].to_numpy()
    assert (np.isnan(rec) == np.isnan(mine)).all(), f"{col}: NaN where the record is not, or the reverse"
    worst = np.nanmax(np.abs(rec - mine), initial=0.0)
    assert worst < 1e-6, f"{col}: the recompute differs from the record by up to {worst:.3g}"
print(f"recompute = record on all {len(D)} keyframes: {', '.join(checks)}")

rng = np.random.default_rng(SEED)
for i in rng.choice(len(D), size=min(200, len(D)), replace=False):
    g = GT_RAW[i][np.isfinite(GT_RAW[i, :, 0])]
    m = score_trajectory(PRED[i], g, interval_s=DT, eval_horizon_s=EVAL_H)
    for k in ("ade", "fde@3s", "cross_track_mae", "ade@10s"):
        rec, got = D[f"rec_{k.replace('@', '_')}"].iat[i], m.get(k, np.nan)
        assert (np.isnan(rec) and np.isnan(got)) or abs(got - rec) < 1e-6, (D.sample_id.iat[i], k, got, rec)
print(f"score_trajectory reproduces the record on {min(200, len(D))} random keyframes")

In [ ]:
def bearing_deg(xy, k=-1):
    """Bearing of waypoint k (default the last, 10 s), degrees left of straight ahead."""
    return np.degrees(np.arctan2(xy[:, k, 1], xy[:, k, 0]))


with np.errstate(invalid="ignore", divide="ignore"):
    chord = np.linalg.norm(GT_RAW[:, -1], axis=-1)
    u = GT_RAW[:, -1] / chord[:, None]
    off_chord = np.abs(GT_RAW[..., 0] * -u[:, None, 1] + GT_RAW[..., 1] * u[:, None, 0]).max(1)
STRAIGHT = (chord >= 100) & (off_chord <= 0.5)
GT_CAL = gt_xy("calibrated")
bearings = pd.DataFrame(
    {
        "vehicle": D.vehicle,
        "mount_yaw_deg": D.yaw_cal_deg,
        "gt_recorded": bearing_deg(GT_RAW),
        "gt_calibrated": bearing_deg(GT_CAL),
        "plan": bearing_deg(PRED),
    }
)[STRAIGHT]
print("straight stretches: mean bearing of the 10 s point (deg, left positive)")
display(
    bearings.groupby("vehicle")
    .agg(n=("plan", "size"), **{c: (c, "mean") for c in bearings.columns[1:]})
    .round(2)
)

fde3 = pd.DataFrame(
    {"vehicle": D.vehicle, "recorded": disp(GT_RAW, n3)[1], "calibrated": disp(GT_CAL, n3)[1]}
)
print(f"FDE@{EVAL_H:g}s (m) against each GT")
display(pd.concat([fde3.assign(vehicle="all"), fde3]).groupby("vehicle").agg(["mean", "median"]).round(3))

if STRAIGHT.any():
    i = int(np.nanargmax(np.where(STRAIGHT, chord, np.nan)))
    fig, ax = plt.subplots(figsize=(5, 7))
    ax.plot(GT_RAW[i, :, 1], GT_RAW[i, :, 0], ":", color=palette.GROUND_TRUTH, lw=1.5, label="GT as recorded")
    bev.draw(ax, {MODEL: PRED[i]}, ground_truth=GT_CAL[i])
    ax.set_aspect("auto")  # stretch the lateral axis: the offset is a degree or two
    ax.legend(loc="upper left", fontsize=8)
    ax.set_title(
        f"longest straight stretch loaded\n{D.sample_id.iat[i]}\nmount yaw {D.yaw_cal_deg.iat[i]:+.2f} deg",
        fontsize=9,
    )
    plt.show()

## 5 Features

`kin()` runs the same kinematics on the GT (in `GT_FRAME`) and on the plan, over the response horizon H (`horizon_s`, columns without a suffix or ending `_H`) and over the intent horizon I (`intent_horizon_s`, 10 s here, columns ending `_I`), with the origin prepended so the first step starts at the car:

- per-step speeds at the step midpoints, giving `v_first`, `v_last`, `v_max`, `v_min` and the mean acceleration `accel`;
- path length `s`;
- heading change `dpsi_deg` between the first and last steps of at least `min_step_m`, and the mean curvature `kappa` over the arc between them;
- `shift`: the lateral offset at the end, net of what a constant-curvature arc with the same heading change explains. It is the lane-change signal, and the only kinematic feature that depends on `GT_FRAME`.

`comfort_H` is paddock's own comfort check (`paddock.metrics.comfort`, nuPlan's bounds) run over [0, H] instead of the recorded 10 s. Over 10 s, openpilot's plan breaks the jerk bound on 70% of moving keyframes, and the GT on 0.2%. Almost all of that sits in the plan's last second, where a few decimetres of noise in positions 300 m out make a large third difference. So the recorded `rec_comfort_*` describe the tail of the plan, not the part a controller would act on.

`features()` then compares the plan with the GT (FDE, along/cross-track error, speed and curvature ratios, the speed the plan starts at, the error in the model's own σ at H). For the radar lead it adds the headway, the lead's own speed `ref_lead_v_abs`, and `gt_x_max_I`, how far forward the driver gets within I. `pred_dpsi_full_deg` is the plan's heading change over its whole 10 s, whatever I is. `FEATURE_DOCS` below documents every column a rule can use.

`v0_ratio` separates two things `speed_ratio` mixes: the speed the model believes the car has now (the plan's first 0.5 s) and what the plan then does with it. The plan starts about 1% faster than the GT on both vehicles, and that offset is most of `too_fast_cruise`; `speed_ratio_net` divides it out. The `start_speed` doc in the tags file has the numbers.

`ref_lead_v_abs` and `gt_x_max_I` exist because the radar's lead is not always a vehicle. It is the closest radar track inside a straight ±1.8 m corridor (`loader.lead_track`), whatever that track is. On a curve the corridor runs into the roadside, so the "lead" can be a sign or a guardrail, and the recorded lead metrics (`rec_lead_*`, `rec_ttc_below_2s`) score the model against it anyway. The `lead` tag calls a near radar lead `clutter` when it is stationary and the driver goes `lead_pass_m` past its range within 10 s, so it was never in the car's path. `lead_motion`, `headway`, `lead_detect`, `lead_range` and `lead_safety` then set it aside, and with them every lead failure mode. See the `lead` doc in the tags file for how much of each mode this was.

In [ ]:
def kin(xy, n, min_step=None):
    """Kinematics of (k, 20, 2) paths over their first n waypoints, as a dict of per-keyframe arrays."""
    min_step = P["min_step_m"] if min_step is None else min_step
    pts = np.concatenate([np.zeros((len(xy), 1, 2)), xy[:, :n]], axis=1)
    step = np.diff(pts, axis=1)
    ds = np.linalg.norm(step, axis=-1)
    full = np.isfinite(ds).all(1)
    rows = np.arange(len(xy))
    with np.errstate(invalid="ignore", divide="ignore"):
        v = np.where(full[:, None], ds / DT, np.nan)
        psi = np.arctan2(step[..., 1], step[..., 0])
        ok = ds >= min_step
        first, last = ok.argmax(1), n - 1 - ok[:, ::-1].argmax(1)
        lat_ok = ok.any(1) & (last > first) & full
        dpsi = np.angle(np.exp(1j * (psi[rows, last] - psi[rows, first])))  # wrapped to (-pi, pi]
        mid = np.cumsum(ds, 1) - ds / 2
        arc = mid[rows, last] - mid[rows, first]
        s = np.where(full, ds.sum(1), np.nan)
        y_end = xy[:, n - 1, 1]
        # A constant-curvature arc of length s turning by dpsi ends (s / dpsi)(1 - cos dpsi) to the side
        arc_y = np.where(np.abs(dpsi) < 1e-6, 0.0, s * (1 - np.cos(dpsi)) / dpsi)
        return {
            "v_first": v[:, 0],
            "v_last": v[:, n - 1],
            "v_max": v.max(1),
            "v_min": v.min(1),
            "accel": (v[:, n - 1] - v[:, 0]) / (max(n - 1, 1) * DT),
            "s": s,
            "dpsi_deg": np.where(lat_ok, np.degrees(dpsi), np.nan),
            "kappa": np.where(lat_ok & (arc > 0), dpsi / arc, np.nan),
            "lat_ok": lat_ok,
            "shift": np.where(lat_ok, y_end - arc_y, np.nan),
        }


def features(gt_frame=None, params=None):
    """D plus the kinematics of GT and plan, and their comparison, per keyframe."""
    p = params or P
    nH, nI = round(p["horizon_s"] / DT), round(p["intent_horizon_s"] / DT)
    gt = gt_xy(gt_frame)
    cols = {}
    for side, xy in (("gt", gt), ("pred", PRED)):
        cols |= {f"{side}_{k}": v for k, v in kin(xy, nH, p["min_step_m"]).items()}
        far = kin(xy, nI, p["min_step_m"])
        cols |= {
            f"{side}_vmin_I": far["v_min"],
            f"{side}_dpsi_I_deg": far["dpsi_deg"],
            f"{side}_shift_I": far["shift"],
        }
        # paddock's comfort bounds over H, not the recorded 10 s (§5)
        comf = pd.DataFrame([comfort(x[:nH], DT) for x in xy])
        cols[f"{side}_comfort_H"] = comf.comfort_within_bounds.to_numpy()
    cols["pred_dpsi_full_deg"] = kin(PRED, N_WP, p["min_step_m"])["dpsi_deg"]
    # the loop ends on the plan, so `comf` is the plan's: which bounds it breaks, and by how much
    cols["pred_comfort_broken_H"] = comf.apply(
        lambda r: " ".join(b for b in BOUNDS if r[f"comfort_{b}_within_bounds"] == 0), axis=1
    ).to_numpy()
    cols["pred_comfort_excess_H"] = (
        pd.concat([comf[f"comfort_{b}_max"] / limit for b, limit in BOUNDS.items()], axis=1)
        .max(axis=1)
        .to_numpy()
    )
    ade, fde = disp(gt, nH)
    along, cross = split(PRED, gt, nH)
    with np.errstate(invalid="ignore", divide="ignore"):
        speed_ratio = np.where(cols["gt_s"] >= p["min_ratio_len_m"], cols["pred_s"] / cols["gt_s"], np.nan)
        # the speed the plan starts at, against the GT's: what the model believes now, apart from what it plans (start_speed)
        v0_ratio = np.where(
            cols["gt_v_first"] >= p["v_go"], cols["pred_v_first"] / cols["gt_v_first"], np.nan
        )
        curving = np.abs(cols["gt_dpsi_deg"]) >= p["curve_min_deg"]
        curv_ratio = np.where(curving, cols["pred_dpsi_deg"] / cols["gt_dpsi_deg"], np.nan)
        sx, sy = SIGMA[:, nH - 1, 0], SIGMA[:, nH - 1, 1]
        dx, dy = (PRED[:, nH - 1] - gt[:, nH - 1]).T
        z = np.maximum(np.abs(dx) / sx, np.abs(dy) / sy)
        headway = np.where(D.ref_speed >= p["v_go"], D.ref_lead_d / D.ref_speed, np.nan)
    # NaN anywhere in the GT's first I seconds leaves this NaN, so the clutter rule it feeds stays False
    reach = gt[:, :nI, 0].max(1)
    cols |= {
        "ade_H": ade,
        "fde_H": fde,
        "along_H": along[:, -1],
        "cross_H": cross[:, -1],
        "speed_ratio": speed_ratio,
        "v0_ratio": v0_ratio,
        "speed_ratio_net": speed_ratio / v0_ratio,
        "curv_ratio": curv_ratio,
        "dpsi_err_deg": cols["pred_dpsi_deg"] - cols["gt_dpsi_deg"],
        "accel_delta": cols["pred_accel"] - cols["gt_accel"],
        "sigma_x_H": sx,
        "sigma_y_H": sy,
        "z_H": z,
        "headway_s": headway,
        "ref_lead_v_abs": (D.ref_speed + D.ref_lead_v_rel).to_numpy(),
        "gt_x_max_I": reach,
    }
    return pd.concat([D, pd.DataFrame(cols, index=D.index)], axis=1)


FEATURE_DOCS = {
    "vehicle, scene, route, chunk, segment, keyframe": "identity; scene = chunk/route/segment, the unit segments are counted in",
    "pitch_cal_deg, yaw_cal_deg": "the route calibration the adapter used (rpy_calib_deg); §4 rotates the GT by yaw_cal_deg",
    "ref_speed, ref_yaw_rate": "CAN speed (m/s) and yaw rate (rad/s, left positive) at the keyframe",
    "ref_lead_d, ref_lead_y, ref_lead_v_rel, ref_has_lead": "the radar's lead: range (m, from the radar), lateral (m), speed relative to ego (m/s, negative = closing)",
    "ref_n_radar_tracks": "radar tracks at the keyframe",
    "op_lead_prob, op_lead_d, op_lead_y, op_lead_v_rel": "the model's lead head, first lead (range from the camera, so a constant offset against the radar is possible)",
    "op_lane_prob": "the model's mean probability of the two ego lane lines",
    "op_v, op_yaw_rate": "the model's pose head: speed (m/s) and yaw rate (rad/s, left positive)",
    "ego_speed_rel_err, ego_yaw_bias": "pose head against the reference: |v_model - v_ref| / max(v_ref, 1), and yaw rate model - reference",
    "rec_*": "every recorded per-keyframe metric with @ written _; rec_fde_3s is FDE@3s against the recorded GT",
    "ctx_*": "the pilot CSV's per-segment context (tags file, context:); NaN outside the pilot segments",
    "{gt,pred}_v_first, _v_last, _v_max, _v_min": "per-step speeds over H (m/s): first step, last step, max, min",
    "{gt,pred}_accel": "(v_last - v_first) / the time between them (m/s²)",
    "{gt,pred}_s": "path length over H (m)",
    "{gt,pred}_dpsi_deg, _kappa, _lat_ok": "heading change over H (deg, left positive) between the first and last steps of at least min_step_m; that over the arc between them (1/m); whether there were two such steps",
    "{gt,pred}_shift": "lateral offset at H net of a constant-curvature arc with the same heading change (m, left positive)",
    "{gt,pred}_vmin_I, _dpsi_I_deg, _shift_I": "the same over intent_horizon_s (I), for the intent tags",
    "pred_dpsi_full_deg": "the plan's heading change over its whole 10 s (deg, left positive), whatever I is",
    "{gt,pred}_comfort_H": "1 within every one of paddock's comfort bounds (paddock.metrics.comfort) over [0, H], 0 not, NaN if unmeasurable; rec_comfort_* are the recorded 10 s version",
    "pred_comfort_broken_H, pred_comfort_excess_H": "the bounds the plan breaks over H, space-separated; its worst peak over that peak's bound (above 1 is broken)",
    "ade_H, fde_H": "ADE / FDE over [0, H] against the GT in GT_FRAME (m)",
    "along_H, cross_H": "along-track (ahead positive) and cross-track (left positive) error at H, against the GT's tangent (m)",
    "speed_ratio": "pred_s / gt_s, the ratio of mean speeds over H; NaN when gt_s < min_ratio_len_m",
    "v0_ratio": "pred_v_first / gt_v_first: the speed the plan starts at against the GT's, i.e. the speed the model believes the car has now; NaN when gt_v_first < v_go",
    "speed_ratio_net": "speed_ratio / v0_ratio: the plan's speed over H relative to where it starts, so a starting-speed offset (start_speed) drops out",
    "curv_ratio": "pred_dpsi_deg / gt_dpsi_deg; NaN unless |gt_dpsi_deg| >= curve_min_deg",
    "dpsi_err_deg": "pred_dpsi_deg - gt_dpsi_deg (deg, left positive): how much more the plan turns than the GT over H, defined on straights too; phantom_curve and early_curve need |dpsi_err_deg| >= curve_err_deg",
    "accel_delta": "pred_accel - gt_accel (m/s²)",
    "sigma_x_H, sigma_y_H, z_H": "the model's own σ at H (m), and the error there in σ: max(|dx|/σx, |dy|/σy)",
    "headway_s": "ref_lead_d / ref_speed (s); NaN below v_go",
    "ref_lead_v_abs": "the radar lead's own speed, ref_speed + ref_lead_v_rel (m/s); about 0 is a stationary object",
    "gt_x_max_I": "the furthest forward the GT gets within intent_horizon_s (m, along the keyframe's x); the `lead` tag compares it with ref_lead_d to tell a lead the driver stops behind from clutter they drive past",
}


def feature_glossary():
    """FEATURE_DOCS as a markdown table."""
    return "\n".join(
        ["| columns | meaning |", "|---|---|", *(f"| {_md(k)} | {_md(v)} |" for k, v in FEATURE_DOCS.items())]
    )


display(Markdown(feature_glossary()))

## 6 Tags

`retag()` computes the features and applies every class, bin, failure mode and named strength in the tags file, giving `T`: `D` plus the features, one column per class (`gt_long`, `pred_long`, `lead_detect`, ...), `fm_<mode>` and `st_<strength>` booleans, and a `sev_<mode>` severity where a mode fired.

- Re-run from here after changing `P`, `GT_FRAME` or the tags file. `retag(params=P | {"a_brake": -0.7})` gives a second table to compare without touching `T`; pass it to any helper as `df=`.
- `select(where=None, df=None, **conds)` returns the matching rows. Each keyword is a column and a value, or a list of values; `where` is any pandas query and can use params as `@name`.
- `explain(sample_id)` shows one keyframe's kinematics and every rule with the values it read. `→` is the rule that set the tag, `✓` a rule that also matched but came later, `·` one that did not match.

The prevalence table is the first sanity check: a tag at 0% or 100% usually means a threshold or a unit is off.

In [ ]:
def _mask(df, expr, p):
    """A boolean Series from a rule. NaN (an undefined comparison) is False."""
    out = df.eval(expr, engine="python", local_dict=p)
    return pd.Series(out, index=df.index).fillna(False).astype(bool)


def _unside(df, side):
    """df with one side's columns also under their bare names (gt_v_max -> v_max), for sided rules."""
    n = len(side) + 1
    return df.assign(**{c[n:]: df[c] for c in df.columns if c.startswith(f"{side}_")})


def _classify(df, spec, p):
    """One class column: the first matching rule's label, `default` if none, `missing` where a required column is NaN."""
    out = pd.Series(spec["default"], index=df.index, dtype=object)
    taken = (
        df[spec["requires"]].isna().any(axis=1) if spec.get("requires") else pd.Series(False, index=df.index)
    )
    out[taken] = spec.get("missing", "na")
    for label, rule in spec["rules"].items():
        hit = _mask(df, rule["when"], p) & ~taken
        out[hit] = label
        taken = taken | hit
    return out


def retag(params=None, gt_frame=None, tags=None, bench=BENCH):
    """The tag table: features, then every class, bin, failure mode and strength that applies to this benchmark."""
    p, tags = params or P, tags or CFG
    df = features(gt_frame, p)
    for name, spec in tags["classes"].items():
        if applies(spec, bench):
            for side in spec.get("sides", [None]):
                df[f"{side}_{name}" if side else name] = _classify(_unside(df, side) if side else df, spec, p)
    for name, spec in tags["bins"].items():
        if applies(spec, bench):
            edges = [p[e[1:]] if isinstance(e, str) else e for e in spec["edges"]]
            cut = pd.cut(df[spec["col"]], [-np.inf, *edges, np.inf], labels=spec["labels"], right=False)
            df[name] = cut.astype(object).where(cut.notna(), spec.get("missing", "na"))
    for kind, prefix in (("failure_modes", "fm_"), ("strengths", "st_")):
        for name, spec in tags[kind].items():
            if applies(spec, bench):
                hit = _mask(df, spec["when"], p)
                df[prefix + name] = hit
                if "severity" in spec:
                    sev = pd.Series(df.eval(spec["severity"], engine="python", local_dict=p), index=df.index)
                    df[f"sev_{name}"] = sev.astype(float).where(hit)
    return df.copy()


ID_TAGS = {"scenario": ["vehicle"]}  # columns of D to condition on beside the tags file's, by group


def tag_columns(df=None, tags=None, bench=BENCH, max_values=12):
    """{group: [columns]}: every categorical column you can condition on, grouped as in the tags file."""
    df, tags = (T if df is None else df), tags or CFG
    groups = {g: list(ID_TAGS.get(g, [])) for g in ("scenario", "intent", "perception", "response")}
    for name, spec in tags["classes"].items():
        if applies(spec, bench):
            for c in _outputs(name, spec):
                groups.setdefault("response" if c.startswith("pred_") else spec["group"], []).append(c)
    for name, spec in tags["bins"].items():
        if applies(spec, bench):
            groups.setdefault(spec["group"], []).append(name)
    groups["failure modes"] = [c for c in df.columns if c.startswith("fm_")]
    groups["strengths"] = [c for c in df.columns if c.startswith("st_")]
    groups["context"] = [c for c in df.columns if c.startswith("ctx_") and df[c].nunique() <= max_values]
    return {g: cs for g, cs in groups.items() if cs}


def select(where=None, df=None, **conds):
    """Rows of T (or df) where every column in conds has the given value (or one of a list), and the query `where`."""
    df = T if df is None else df
    unknown = [c for c in conds if c not in df.columns]
    if unknown:
        known = [c for cs in tag_columns(df).values() for c in cs]
        raise KeyError(f"{unknown} are not columns; the tag columns are {known}")
    mask = pd.Series(True, index=df.index)
    for col, val in conds.items():
        mask &= df[col].isin(val if isinstance(val, (list, tuple, set)) else [val])
    out = df[mask]
    return out.query(where, engine="python", local_dict=P) if where else out


def prevalence(df=None, groups=None):
    """Share of keyframes for each value of each tag column."""
    df = T if df is None else df
    rows = []
    for g, cols in (groups or tag_columns(df)).items():
        for c in cols:
            if df[c].dtype == bool:
                values = f"{df[c].mean():.1%} ({int(df[c].sum())})"
            else:
                vc = df[c].value_counts(normalize=True, dropna=False)
                values = " · ".join(f"{k} {v:.0%}" for k, v in vc.items())
            rows.append({"group": g, "tag": c, "values": values})
    return pd.DataFrame(rows).set_index(["group", "tag"])


_KEYWORDS = {"and", "or", "not", "in", "abs", "True", "False"}


def _names(expr):
    """The column names a rule reads: identifiers that are not @params, keywords or quoted strings."""
    bare = re.sub(r"'[^']*'|\"[^\"]*\"", "", expr or "")
    return [n for n in dict.fromkeys(re.findall(r"(?<![@\w])[A-Za-z_]\w*", bare)) if n not in _KEYWORDS]


def _fmt(v):
    return f"{v:.3g}" if isinstance(v, (float, np.floating)) else str(v)


KIN = [
    "v_first",
    "v_last",
    "v_max",
    "v_min",
    "accel",
    "s",
    "dpsi_deg",
    "kappa",
    "lat_ok",
    "shift",
    "vmin_I",
    "dpsi_I_deg",
    "shift_I",
]
COMPARED = [
    "fde_H",
    "ade_H",
    "along_H",
    "cross_H",
    "speed_ratio",
    "v0_ratio",
    "speed_ratio_net",
    "curv_ratio",
    "dpsi_err_deg",
    "accel_delta",
    "z_H",
    "rec_fde_3s",
    "ref_speed",
    "headway_s",
]


def explain(sample_id, df=None, params=None, tags=None, bench=BENCH):
    """Why each tag and mode fired for one keyframe: its kinematics, then every rule with the values it read."""
    p, tags = params or P, tags or CFG
    df = T if df is None else df
    one = df.loc[[_row(sample_id, df)]]
    print(one.sample_id.iat[0])
    display(
        pd.DataFrame(
            {side: [float(one[f"{side}_{k}"].iat[0]) for k in KIN] for side in ("gt", "pred")}, index=KIN
        ).T.round(3)
    )
    display(one[COMPARED].round(3))
    lines = []
    for name, spec in tags["classes"].items():
        if not applies(spec, bench):
            continue
        for side in spec.get("sides", [None]):
            col = f"{side}_{name}" if side else name
            sub = _unside(one, side) if side else one
            label = one[col].iat[0]
            lines.append(f"{col} = {label}")
            if spec.get("requires") and sub[spec["requires"]].isna().any(axis=1).iat[0]:
                lines.append(f"    NaN in {spec['requires']} -> {spec.get('missing', 'na')}")
                continue
            for rule_label, rule in spec["rules"].items():
                fired = bool(_mask(sub, rule["when"], p).iat[0])
                mark = "→" if rule_label == label else "✓" if fired else "·"
                values = ", ".join(f"{n}={_fmt(sub[n].iat[0])}" for n in _names(rule["when"]))
                lines.append(f"  {mark} {rule_label:<14} {rule['when']:<45} {values}")
    print("\n".join(lines))
    fired = [c for c in one.columns if c.startswith("fm_") and one[c].iat[0]]
    for c in fired:
        spec = tags["failure_modes"][c[3:]]
        values = ", ".join(f"{n}={_fmt(one[n].iat[0])}" for n in _names(spec["when"]))
        print(f"MODE {c[3:]} (severity {_fmt(one[f'sev_{c[3:]}'].iat[0])}): {spec['when']}\n    {values}")
    quiet = [c[3:] for c in one.columns if c.startswith("fm_") and c not in fired]
    print(textwrap.fill("did not fire: " + ", ".join(quiet), 110))
    strong = [c[3:] for c in one.columns if c.startswith("st_") and one[c].iat[0]]
    print("strengths: " + (", ".join(strong) or "none"))


def _row(key, df):
    """A row label (= array position) from a sample id, or a row label passed through."""
    if isinstance(key, str):
        hit = df.index[df.sample_id == key]
        if not len(hit):
            raise KeyError(f"{key!r} is not among the loaded keyframes (FOCUS / SUBSET in §1)")
        return hit[0]
    return key


T = retag()
print(f"T: {len(T)} keyframes x {T.shape[1]} columns, GT frame {GT_FRAME!r}")
with pd.option_context("display.max_colwidth", None, "display.max_rows", 200):
    display(prevalence())

In [ ]:
explain(T.sample_id.iat[0])

**Trying a rule change without editing the file.** Work on a copy of `CFG` and pass it as `tags=`; every helper then takes the new table as `df=`:

```python
import copy
CFG2 = copy.deepcopy(CFG)
CFG2["classes"]["long"]["rules"]["brake"]["when"] = "accel <= -0.3"
CFG2["failure_modes"]["slow_on_curves"] = {"group": "long", "when": "gt_lat in ['curve_L', 'curve_R'] and speed_ratio < 0.9", "severity": "1 - speed_ratio", "doc": "..."}
T2 = retag(tags=CFG2)
confusion("long", df=T2); modes(df=T2, tags=CFG2)
```

## 7 Scores

All recomputed in numpy from `PRED` and the GT arrays, so any value can be tried in well under a second, even on every keyframe.

1. `recorded()`: the run's `metrics.json` exactly as stored, overall and per vehicle. That is against the **recorded** GT, over the whole run, whatever is loaded here.
2. `scores(by=, horizons=, gt_frame=, yaw_offset_deg=, miss_m=, stat=, where=, **conds)`: ADE, FDE, miss rate (FDE > `miss_m`), along-track error and |cross-track| error at each horizon, per group of any tag column(s), on the loaded keyframes. `by_frame()` puts both GT frames side by side.
3. `score_explorer()`: the same behind widgets, with `WIDGETS = True` (§1).
4. `fde_curve()`: a metric against horizon, one line per group.
5. `yaw_sweep()`: FDE at H against an extra rotation of the GT. For the calibrated GT the minimum should sit near 0°; for the recorded GT near each vehicle's mean mount yaw (dotted). A minimum elsewhere would mean the calibration does not explain the offset.

With every keyframe loaded, the recorded-frame recompute is asserted equal to `metrics.json`.

In [ ]:
def _hkey(k):
    name, _, h = k.partition("@")
    return (name, float(h.rstrip("s")) if h else EVAL_H)


def recorded(pattern=r"ade(@\d+s)?|fde@\d+s|miss_rate@\d+s", stat="mean"):
    """metrics.json as stored (recorded GT, whole run), overall and per vehicle: `stat` of every metric matching pattern."""
    groups = {"all": {"n": M["n"], **M["metrics"]}, **M.get("by_split", {})}
    rows = {
        g: {"n": m.get("n")}
        | {k: v[stat] for k, v in m.items() if isinstance(v, dict) and re.fullmatch(pattern, k)}
        for g, m in groups.items()
    }
    out = pd.DataFrame(rows).T
    return out[["n", *sorted(out.columns.drop("n"), key=_hkey)]].round(3)


def per_keyframe(horizons=(1, 2, 3, 5, 10), gt_frame=None, yaw_offset_deg=0.0, miss_m=None):
    """Per keyframe: ADE, FDE, miss, along and |cross| at each horizon (s), against the GT in gt_frame."""
    gt, miss_m = gt_xy(gt_frame, yaw_offset_deg), P["miss_m"] if miss_m is None else miss_m
    cols = {}
    for h in horizons:
        n = round(h / DT)
        ade, fde = disp(gt, n)
        along, cross = split(PRED, gt, n)
        cols |= {
            f"ade@{h:g}s": ade,
            f"fde@{h:g}s": fde,
            f"miss@{h:g}s": np.where(np.isnan(fde), np.nan, fde > miss_m),
            f"along@{h:g}s": along[:, -1],
            f"|cross|@{h:g}s": np.abs(cross[:, -1]),
        }
    return pd.DataFrame(cols, index=D.index)


STATS = {"median": "median", "mean": "mean", "p90": lambda s: s.quantile(0.9)}


def scores(
    by=None,
    horizons=(1, 2, 3, 5, 10),
    gt_frame=None,
    yaw_offset_deg=0.0,
    miss_m=None,
    stat="median",
    where=None,
    df=None,
    **conds,
):
    """Errors per group of any tag column(s), on the loaded keyframes. Miss rates are always a mean."""
    sel = select(where, df, **conds)
    k = per_keyframe(horizons, gt_frame, yaw_offset_deg, miss_m).loc[sel.index]
    keys = [by] if isinstance(by, str) else list(by or [])
    grouped = k.assign(**{c: sel[c] for c in keys}, _scene=sel.scene).groupby(
        keys or (lambda _: "all"), dropna=False, observed=True
    )
    miss = [c for c in k if c.startswith("miss@")]
    out = grouped[[c for c in k if c not in miss]].agg(STATS[stat])
    out[miss] = grouped[miss].mean()
    out.insert(0, "segments", grouped["_scene"].nunique())
    out.insert(0, "n", grouped.size())
    return out[["n", "segments", *k.columns]].round(3)


def by_frame(
    by="vehicle", stat="mean", horizons=(1, 2, 3, 5, 10), cols=r"^(n|segments|ade@3s|fde@|miss@)", **kw
):
    """scores() against both GT frames side by side (columns: frame, metric)."""
    parts = {f: scores(by, horizons, gt_frame=f, stat=stat, **kw) for f in ("recorded", "calibrated")}
    out = pd.concat(parts, axis=1)
    return out.loc[:, [c for c in out.columns if re.match(cols, c[1])]]


print("recorded: metrics.json, mean over the whole run, recorded GT")
display(recorded())
print(f"recomputed on the {len(D)} loaded keyframes, mean, both GT frames")
display(by_frame())
if len(D) == M["n"]:
    whole = per_keyframe((1, 2, 3), gt_frame="recorded").mean()
    for mine, rec in (("ade@3s", "ade"), ("fde@1s", "fde@1s"), ("fde@2s", "fde@2s"), ("fde@3s", "fde@3s")):
        assert abs(whole[mine] - M["metrics"][rec]["mean"]) < 1e-5, (
            mine,
            whole[mine],
            M["metrics"][rec]["mean"],
        )
    print("every keyframe loaded: the recorded-frame recompute equals metrics.json")

In [ ]:
display(scores(by="gt_long"))
display(scores(by=["vehicle", "speed_band"], gt_lat="straight", stat="mean"))

In [ ]:
def _as_html(fn, *args, **kw):
    """Run fn; return what it printed and displayed, in order, as HTML for a W.HTML widget.

    The widgets render into W.HTML rather than W.Output so that a headless run cannot hang.
    nbclient (what `jupyter execute` runs) emulates an Output widget by sending its contents
    back to the kernel on the socket it is polling for the cell's reply; for some sizes that
    reply is then missed, and the cell waits out its whole timeout -- which `jupyter execute`
    does not set. A W.HTML's state only ever travels kernel -> frontend: nothing to miss.
    """
    shell, parts = get_ipython(), []

    class Stdout(io.TextIOBase):
        def write(self, s):
            parts.append(s)
            return len(s)

    saved, shell.display_pub = shell.display_pub, DisplayPublisher()
    shell.display_pub.publish = lambda data, *a, **k: parts.append(data)
    try:
        with contextlib.redirect_stdout(Stdout()):
            fn(*args, **kw)
    except Exception as e:  # an empty selection or a bad query: show it rather than break the widget
        parts.append(f"{type(e).__name__}: {e}\n")
    finally:
        shell.display_pub = saved
    blocks = []
    for is_text, group in itertools.groupby(parts, key=lambda p: isinstance(p, str)):
        if is_text:  # consecutive prints share one block
            blocks.append(f"<pre>{html.escape(''.join(group))}</pre>")
            continue
        for data in group:
            mime = next((m for m in ("image/png", "image/jpeg") if m in data), None)
            if mime:
                img = data[mime] if isinstance(data[mime], str) else base64.b64encode(data[mime]).decode()
                blocks.append(f'<img src="data:{mime};base64,{img}" style="max-width:100%">')
            else:
                blocks.append(
                    data.get("text/html") or f"<pre>{html.escape(data.get('text/plain', ''))}</pre>"
                )
    return "\n".join(blocks)


def _condition_widgets(df, groups=None):
    """One multi-select per tag column (every value, with its count), one tab per group. Returns (tab, pickers, conds)."""
    groups = groups or tag_columns(df)
    pickers, pages = {}, []
    for cols in groups.values():
        boxes = []
        for c in cols:
            vc = df[c].value_counts(dropna=False)
            pickers[c] = W.SelectMultiple(
                options=[(f"{v} ({n})", v) for v, n in vc.items()],
                rows=min(6, len(vc)),
                layout=W.Layout(width="300px"),
            )
            boxes.append(W.VBox([W.Label(c), pickers[c]]))
        pages.append(W.GridBox(boxes, layout=W.Layout(grid_template_columns="repeat(3, 310px)")))
    tab = W.Tab(children=pages, titles=list(groups))

    def conds():
        return {
            c: [v.item() if hasattr(v, "item") else v for v in w.value] for c, w in pickers.items() if w.value
        }

    return tab, pickers, conds


def _no_widgets(instead):
    """True, after saying what to call instead, unless WIDGETS (§1) is on and ipywidgets is installed."""
    if WIDGETS and W is not None:
        return False
    print(f"{'ipywidgets is not installed' if WIDGETS else 'widgets are off (WIDGETS in §1)'}: {instead}")
    return True


def score_explorer(df=None):
    """scores() behind widgets: group by, horizons, GT frame, extra yaw, miss radius, statistic and conditions."""
    if _no_widgets("call scores(...) directly"):
        return
    df = T if df is None else df
    cols = [c for cs in tag_columns(df).values() for c in cs]
    by = W.Dropdown(options=[None, *cols], value="vehicle", description="by")
    then = W.Dropdown(options=[None, *cols], value=None, description="then by")
    hz = W.SelectMultiple(
        options=[0.5, 1, 2, 3, 4, 5, 6, 8, 10], value=(1, 2, 3, 5, 10), rows=9, description="horizons (s)"
    )
    frame = W.ToggleButtons(options=["calibrated", "recorded"], value=GT_FRAME, description="GT frame")
    yaw = W.FloatSlider(value=0.0, min=-3, max=3, step=0.1, description="+ yaw (deg)")
    miss = W.FloatSlider(value=P["miss_m"], min=0.5, max=5, step=0.25, description="miss (m)")
    stat = W.Dropdown(options=list(STATS), value="median", description="stat")
    tab, pickers, conds = _condition_widgets(df)
    out = W.HTML()

    def table():
        keys = [k for k in (by.value, then.value) if k]
        display(
            scores(keys, tuple(hz.value), frame.value, yaw.value, miss.value, stat.value, df=df, **conds())
        )

    def refresh(*_):
        out.value = _as_html(table)

    for w in (by, then, hz, frame, yaw, miss, stat, *pickers.values()):
        w.observe(refresh, "value")
    refresh()
    display(W.VBox([W.HBox([by, then, stat]), W.HBox([hz, W.VBox([frame, yaw, miss])]), tab, out]))


score_explorer()

In [ ]:
def fde_curve(
    by="vehicle",
    metric="fde",
    gt_frame=None,
    stat="median",
    horizons=(0.5, 1, 1.5, 2, 3, 4, 5, 6, 8, 10),
    where=None,
    df=None,
    **conds,
):
    """A metric (ade, fde, miss, along, |cross|) against horizon, one line per group."""
    s = scores(by, horizons, gt_frame, stat=stat, where=where, df=df, **conds)
    cols = [f"{metric}@{h:g}s" for h in horizons]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for key, row in s.iterrows():
        ax.plot(horizons, row[cols].to_numpy(float), marker="o", ms=3, label=f"{key} (n={row.n:.0f})")
    ax.set(
        xlabel="horizon (s)",
        ylabel=f"{stat} {metric} (m)",
        title=f"{metric} against horizon, GT {gt_frame or GT_FRAME}",
    )
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()
    return s[["n", *cols]]


def yaw_sweep(offsets=None, h_s=None, by="vehicle", stat="mean", where=None, df=None, **conds):
    """FDE at h_s against an extra GT yaw, in both frames. Returns the yaw of the minimum per frame and group."""
    offsets = np.linspace(-3, 3, 25) if offsets is None else np.asarray(offsets)
    h_s = h_s or EVAL_H
    col = f"fde@{h_s:g}s"
    sel = select(where, df, **conds)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    best = {}
    for ax, frame in zip(axes, ("recorded", "calibrated")):
        curves = pd.DataFrame(
            {o: scores(by, (h_s,), frame, o, stat=stat, where=where, df=df, **conds)[col] for o in offsets}
        )
        for key, row in curves.iterrows():
            (line,) = ax.plot(offsets, row.to_numpy(float), label=str(key))
            best[(frame, key)] = offsets[np.nanargmin(row.to_numpy(float))]
            if frame == "recorded" and by == "vehicle":
                ax.axvline(
                    sel.loc[sel.vehicle == key, "yaw_cal_deg"].mean(), color=line.get_color(), ls=":", lw=1
                )
        ax.axvline(0, color="k", lw=0.6)
        ax.set(title=f"GT {frame}", xlabel="extra GT yaw (deg, left positive)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    axes[0].set_ylabel(f"{stat} {col} (m)")
    plt.show()
    return pd.Series(best).unstack().rename_axis(index="frame")


display(fde_curve())
display(yaw_sweep())

## 8 Distributions, with the thresholds drawn in

`dist(cols=, by=, bins=, params=, where=, **conds)` draws one histogram (density) per column and group, with every threshold the tags file applies to that column as a dashed line. This is where a guessed threshold gets tuned: a threshold in the middle of a dense mode splits one behaviour into two tags. Values beyond the 1st–99th percentile range (widened to include the thresholds) are piled into the end bins. `THRESHOLDS` maps each column to the params drawn on it; `-name` draws the negated value.

`scatter()` puts the GT against the plan for the same feature. Points on the dashed diagonal are plans that did what the GT did.

In [ ]:
THRESHOLDS = {
    "gt_accel": ["a_hard_brake", "a_brake", "a_accel", "a_hard_accel"],
    "pred_accel": ["a_hard_brake", "a_brake", "a_accel", "a_hard_accel"],
    "gt_kappa": ["-k_sharp", "-k_straight", "k_straight", "k_sharp"],
    "pred_kappa": ["-k_sharp", "-k_straight", "k_straight", "k_sharp"],
    "gt_dpsi_deg": ["-turn_deg", "-curve_min_deg", "curve_min_deg", "turn_deg"],
    "speed_ratio": ["ratio_much_slow", "ratio_slow", "ratio_fast", "ratio_much_fast"],
    "v0_ratio": ["start_slow", "start_fast"],
    "speed_ratio_net": ["ratio_much_slow", "ratio_slow", "ratio_fast", "ratio_much_fast"],
    "curv_ratio": [0.0, "curv_under", "curv_over"],
    "dpsi_err_deg": ["-curve_err_deg", "curve_err_deg"],
    "accel_delta": ["-accel_tol", "accel_tol"],
    "cross_H": ["-drift_m", "-side_m", "side_m", "drift_m"],
    "z_H": ["z1", "z2", "z_bad"],
    "fde_H": ["fde_bad_m", "miss_m"],
    "ref_speed": ["v_stop", "v_low", "v_mid"],
    "headway_s": ["headway_tailgate_s", "headway_close_s"],
    "ref_lead_v_rel": ["closing_mps", "opening_mps"],
    "ref_yaw_rate": ["-yaw_turning", "yaw_turning"],
    "op_lane_prob": ["lane_none", "lane_weak"],
    "ego_speed_rel_err": ["speed_read_tol"],
    "ego_yaw_bias": ["-yaw_bias_tol", "yaw_bias_tol"],
    "rec_lead_range_err": ["-lead_range_tol_m", "lead_range_tol_m"],
    "gt_shift_I": ["-lc_max_m", "-lc_min_m", "lc_min_m", "lc_max_m"],
}


def _lines(col, p):
    """[(label, value)] of the thresholds drawn on a column."""
    out = []
    for t in THRESHOLDS.get(col, []):
        if isinstance(t, str):
            name = t.lstrip("-")
            out.append((t, -p[name] if t.startswith("-") else p[name]))
        else:
            out.append((f"{t:g}", t))
    return out


def _groups(sel, by):
    return sel.groupby(by, dropna=False, observed=True) if by else [("all", sel)]


def dist(cols=None, by="vehicle", bins=60, params=None, where=None, df=None, **conds):
    """Histograms per group with the tag thresholds as dashed lines."""
    p = params or P
    sel = select(where, df, **conds)
    cols = cols or list(THRESHOLDS)
    ncol = min(3, len(cols))
    nrow = -(-len(cols) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3 * nrow), squeeze=False)
    for ax, col in zip(axes.flat, cols):
        x = sel[col].astype(float)
        x = x[np.isfinite(x)]
        if x.empty:
            ax.set_title(f"{col}: no values", fontsize=9)
            continue
        th = [v for _, v in _lines(col, p)]
        lo, hi = np.percentile(x, [1, 99])
        lo, hi = min([lo, *th]), max([hi, *th])
        if hi <= lo:
            lo, hi = lo - 1, hi + 1
        edges = np.linspace(lo, hi, bins + 1)
        for key, grp in _groups(sel, by):
            v = grp[col].astype(float)
            v = v[np.isfinite(v)].clip(lo, hi)
            if len(v):
                ax.hist(v, edges, density=True, histtype="step", lw=1.3, label=f"{key} ({len(v)})")
        for name, v in _lines(col, p):
            ax.axvline(v, color="k", ls="--", lw=0.8, alpha=0.6)
            ax.text(v, ax.get_ylim()[1], f" {name}", rotation=90, va="top", ha="right", fontsize=7)
        ax.set_title(col, fontsize=9)
        ax.legend(fontsize=7)
    for ax in axes.flat[len(cols) :]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()


def scatter(
    pairs=(
        ("gt_accel", "pred_accel"),
        ("gt_kappa", "pred_kappa"),
        ("gt_s", "pred_s"),
        ("gt_dpsi_deg", "pred_dpsi_deg"),
    ),
    by="vehicle",
    params=None,
    where=None,
    df=None,
    **conds,
):
    """GT (x) against plan (y) for each pair, y = x dashed, the GT column's thresholds dotted on both axes."""
    p = params or P
    sel = select(where, df, **conds)
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.3 * len(pairs), 4.2), squeeze=False)
    for ax, (xc, yc) in zip(axes[0], pairs):
        for key, grp in _groups(sel, by):
            ax.scatter(grp[xc], grp[yc], s=4, alpha=0.4, label=f"{key} ({len(grp)})")
        xy = sel[[xc, yc]].astype(float).to_numpy()
        xy = xy[np.isfinite(xy).all(1)]
        if len(xy):
            lo, hi = np.percentile(xy, [0.5, 99.5])
            ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
            ax.set(xlim=(lo, hi), ylim=(lo, hi))
        for _, v in _lines(xc, p):
            ax.axvline(v, color="grey", lw=0.6, ls=":")
            ax.axhline(v, color="grey", lw=0.6, ls=":")
        ax.set(xlabel=xc, ylabel=yc)
        ax.legend(fontsize=7, markerscale=3)
    fig.tight_layout()
    plt.show()


dist()
scatter()

## 9 Scenario clusters

`clusters(by=("gt_lat", "gt_long", "speed_band"), min_n=10, where=, **conds)` groups keyframes by any combination of tags. Per cluster: the number of keyframes and segments, median and p90 FDE at H, the share where the plan's speed and curvature match the GT (`speed_ratio_bin` / `curv_ratio_bin` = matched, among keyframes where they are defined), the plan's most common response, and the failure mode with the highest rate there.

`heatmap(row=, col=, value=, stat=, min_n=)` shows one value over two tags, e.g. the median `fde_H`, or the rate of a failure mode (`value="fm_under_brakes"`). Cells under `min_n` keyframes are blank.

In [ ]:
ORDER = {  # reading order for tables and matrices; other tags follow the tags file
    "long": ["stationary", "starting", "stopping", "hard_brake", "brake", "cruise", "accel", "hard_accel"],
    "lat": ["turn_L", "sharp_L", "curve_L", "straight", "curve_R", "sharp_R", "turn_R", "na"],
}


def _order(col, values, tags=None):
    """values in the tag's natural order (ORDER, else the tags file's rule order), unknown ones last."""
    tags = tags or CFG
    bare = re.sub(r"^(gt|pred)_", "", col)
    spec = tags["classes"].get(bare) or tags["bins"].get(bare) or {}
    key = ORDER.get(bare) or [
        *spec.get("rules", {}),
        spec.get("default"),
        *spec.get("labels", []),
        spec.get("missing", "na"),
    ]
    return sorted(values, key=lambda v: (key.index(v) if v in key else len(key), str(v)))


def _top(s):
    vc = s.value_counts(normalize=True)
    return f"{vc.index[0]} {vc.iloc[0]:.0%}" if len(vc) else ""


def _matched(s):
    """1.0 where a ratio bin is matched, 0.0 where it is not, NaN where it is undefined."""
    return s.eq("matched").astype(float).where(s.ne("na"))


def _intent(col, tags=None):
    return (tags or CFG)["failure_modes"].get(col[3:], {}).get("group") == "intent"


def clusters(by=("gt_lat", "gt_long", "speed_band"), min_n=10, where=None, df=None, **conds):
    """Every combination of the `by` tags with at least min_n keyframes, and how the plan fared there."""
    sel = select(where, df, **conds)
    fm = [c for c in sel.columns if c.startswith("fm_") and not _intent(c)]
    g = sel.assign(_speed=_matched(sel.speed_ratio_bin), _curv=_matched(sel.curv_ratio_bin)).groupby(
        list(by), dropna=False, observed=True
    )
    out = g.agg(
        n=("sample_id", "size"),
        segments=("scene", "nunique"),
        fde_H_median=("fde_H", "median"),
        fde_H_p90=("fde_H", lambda s: s.quantile(0.9)),
        speed_matched_pct=("_speed", "mean"),
        curv_matched_pct=("_curv", "mean"),
        pred_long=("pred_long", _top),
        pred_lat=("pred_lat", _top),
    )
    out[["speed_matched_pct", "curv_matched_pct"]] *= 100
    rates = g[fm].mean()
    out["top_mode"] = [
        f"{c[3:]} {r[c]:.0%}" if r[c] > 0 else "" for c, (_, r) in zip(rates.idxmax(axis=1), rates.iterrows())
    ]
    return out[out.n >= min_n].sort_values("n", ascending=False).round(2)


def heatmap(row="gt_long", col="gt_lat", value="fde_H", stat="median", min_n=5, where=None, df=None, **conds):
    """One value over two tags, annotated with n; a boolean value (fm_*, st_*) shows its rate."""
    sel = select(where, df, **conds)
    how = "mean" if sel[value].dtype == bool else stat
    g = sel.assign(_v=sel[value].astype(float)).groupby([row, col], observed=True)["_v"]
    rows, cols = _order(row, sel[row].unique()), _order(col, sel[col].unique())
    t = g.agg(STATS[how]).unstack().reindex(index=rows, columns=cols)
    n = g.size().unstack().reindex(index=rows, columns=cols).fillna(0).astype(int)
    t = t.where(n >= min_n)
    fig, ax = plt.subplots(figsize=(1.15 * len(cols) + 2.5, 0.55 * len(rows) + 1.6))
    im = ax.imshow(t.to_numpy(float), cmap="viridis", aspect="auto")
    for (r, c), v in np.ndenumerate(t.to_numpy(float)):
        text = f"{v:.2f}\nn={n.iat[r, c]}" if np.isfinite(v) else (f"n={n.iat[r, c]}" if n.iat[r, c] else "")
        ax.text(c, r, text, ha="center", va="center", fontsize=7, color="w" if np.isfinite(v) else "grey")
    ax.set_xticks(range(len(cols)), cols, rotation=30, ha="right")
    ax.set_yticks(range(len(rows)), rows)
    ax.set(xlabel=col, ylabel=row, title=f"{how} {value} (blank: n < {min_n})")
    fig.colorbar(im, ax=ax)
    plt.show()
    return t.round(3)


with pd.option_context("display.max_rows", 60):
    display(clusters())
display(heatmap())
display(heatmap("gt_lat", "speed_band", value="speed_ratio"))

## 10 Confusion: GT class against plan class

`confusion(axis="long" | "lat", normalize="index")`: rows are the GT's class over H, columns the plan's, from the same rules. With `normalize="index"` each row sums to 1 and reads as "when the GT did this, the plan did …"; `"columns"` reads as "when the plan did this, the GT had done …"; `"all"` or `None` for shares or counts. Cells show the share and the count.

In [ ]:
def confusion(axis="long", normalize="index", where=None, df=None, **conds):
    """GT class (rows) against plan class (columns), annotated with share and count. Returns the counts."""
    sel = select(where, df, **conds)
    gt, pred = sel[f"gt_{axis}"], sel[f"pred_{axis}"]
    labels = _order(axis, set(gt) | set(pred))
    counts = pd.crosstab(gt, pred).reindex(index=labels, columns=labels, fill_value=0)
    if normalize == "index":
        share = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)
    elif normalize == "columns":
        share = counts.div(counts.sum(axis=0).replace(0, np.nan), axis=1)
    elif normalize == "all":
        share = counts / max(counts.to_numpy().sum(), 1)
    else:
        share = counts.astype(float)
    fig, ax = plt.subplots(figsize=(0.95 * len(labels) + 2.5, 0.8 * len(labels) + 1.5))
    ax.imshow(share.to_numpy(float), cmap="Blues", vmin=0)
    top = np.nanmax(share.to_numpy(float), initial=0.0)
    for (r, c), v in np.ndenumerate(share.to_numpy(float)):
        if counts.iat[r, c]:
            text = f"{v:.0%}\n{counts.iat[r, c]}" if normalize else f"{counts.iat[r, c]}"
            ax.text(c, r, text, ha="center", va="center", fontsize=7, color="w" if v > 0.6 * top else "k")
    ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set(xlabel=f"plan: pred_{axis}", ylabel=f"GT: gt_{axis}", title=f"{axis}, normalised by {normalize}")
    plt.show()
    return counts


display(confusion("long"))
display(confusion("lat"))

## 11 Conditions: `profile()` and `explore()`

Two equivalent ways to pick keyframes and see what they look like.

- **Variables.** Set `COND` (any tag column: a value or a list of values) and `WHERE` (any pandas query) and call `profile(where=WHERE, **COND)`. `select()` with the same arguments returns the rows themselves.
- **Widgets.** With `WIDGETS = True` (§1), `explore()` has one multi-select per tag column (every observed value with its count), range sliders for speed, the ratios and the errors, and a query box. It re-runs `profile()` on every change and prints the `COND` and `WHERE` it applied, so a selection made with the mouse can be pasted into the cell above and kept. The button draws three matches with `show()`.

`profile()` shows: the size of the selection; how the plan responded; the error and ratio columns; the failure modes with their rate against the rate over everything loaded (lift); the scenario, perception and context tags most over-represented in the selection (lift, excluding the columns you conditioned on); the selection's histograms against everything; and sample ids to look at, one per segment.

In [ ]:
def lift(sel, base=None, groups=("scenario", "intent", "perception", "context"), exclude=(), top=10, min_n=3):
    """Tag values over-represented in sel: share in sel / share in base (default all of T)."""
    base = T if base is None else base
    cols = [c for g, cs in tag_columns(base).items() if g in groups for c in cs if c not in exclude]
    rows = []
    for c in cols:
        # shares among the keyframes where the column is defined: ctx_* are NaN outside the pilot segments, and
        # counting NaN as a value would make a context lift measure how much of the selection is in the pilot
        in_base = base[c].value_counts(normalize=True)
        known = sel[c].notna().sum()
        for v, k in sel[c].value_counts().items():
            if k >= min_n and v in in_base.index:
                share = k / known
                rows.append(
                    {
                        "tag": c,
                        "value": v,
                        "n": k,
                        "share": share,
                        "base_share": in_base[v],
                        "lift": share / in_base[v],
                    }
                )
    if not rows:
        return pd.DataFrame(columns=["tag", "value", "n", "share", "base_share", "lift"])
    return pd.DataFrame(rows).sort_values("lift", ascending=False).head(top).round(3).reset_index(drop=True)


def _picks(sel, k, seed=None):
    """Up to k sample ids, at most one per segment, the same ones for the same seed."""
    seed = SEED if seed is None else seed
    if sel.empty:
        return []
    one = sel.groupby("scene").sample(1, random_state=seed)
    return one.sample(min(k, len(one)), random_state=seed).sample_id.tolist()


def profile(where=None, df=None, top=10, n_ids=6, seed=None, **conds):
    """What a selection of keyframes looks like, against everything loaded."""
    base = T if df is None else df
    sel = select(where, base, **conds)
    print(
        f"{len(sel)} keyframes ({len(sel) / len(base):.1%}); {sel.scene.nunique()} of {base.scene.nunique()} segments; "
        f"{sel.route.nunique()} of {base.route.nunique()} routes"
    )
    if sel.empty:
        return
    display(prevalence(sel, {"response": tag_columns(base).get("response", [])}))
    display(
        sel[["fde_H", "cross_H", "speed_ratio", "curv_ratio", "accel_delta", "z_H", "rec_fde_3s"]]
        .describe()
        .T.round(3)
    )
    fm = [c for c in base.columns if c.startswith("fm_")]
    rates = pd.DataFrame({"n": sel[fm].sum(), "rate": sel[fm].mean(), "base_rate": base[fm].mean()})
    rates["lift"] = rates.rate / rates.base_rate
    display(rates[rates.n > 0].sort_values("n", ascending=False).round(3))
    display(lift(sel, base, exclude=[*conds, *_names(where)], top=top))
    fig, axes = plt.subplots(1, 4, figsize=(16, 2.8))
    for ax, c in zip(axes, ("fde_H", "speed_ratio", "curv_ratio", "cross_H")):
        a, b = base[c].astype(float), sel[c].astype(float)
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        if len(a):
            lo, hi = np.percentile(a, [1, 99])
            edges = np.linspace(lo, hi if hi > lo else lo + 1, 41)
            ax.hist(
                a.clip(lo, hi), edges, density=True, histtype="step", color="grey", label=f"all ({len(a)})"
            )
            if len(b):
                ax.hist(
                    b.clip(lo, hi),
                    edges,
                    density=True,
                    histtype="step",
                    lw=1.5,
                    label=f"selection ({len(b)})",
                )
            ax.legend(fontsize=7)
        ax.set_title(c, fontsize=9)
    fig.tight_layout()
    plt.show()
    print("to look at, one per segment: show('<id>')\n" + "\n".join(_picks(sel, n_ids, seed)))

In [ ]:
COND = dict(gt_lat="curve_R", gt_long="cruise")  # any tag column: a value or a list of values
WHERE = ""  # optional pandas query, e.g. "speed_ratio < 0.9 and ref_speed > @v_mid"
profile(where=WHERE or None, **COND)

In [ ]:
RANGES = ("ref_speed", "speed_ratio", "curv_ratio", "fde_H", "cross_H", "z_H")


def explore(df=None):
    """profile() behind widgets: tag multi-selects, range sliders and a query box. Prints the COND and WHERE to paste."""
    if _no_widgets("set COND and WHERE and call profile(...)"):
        return
    df = T if df is None else df
    tab, pickers, conds = _condition_widgets(df)
    sliders = {}
    for c in RANGES:
        x = df[c].astype(float)
        x = x[np.isfinite(x)]
        if len(x):
            lo, hi = float(x.quantile(0.005)), float(x.quantile(0.995))
            sliders[c] = W.FloatRangeSlider(
                value=(lo, hi),
                min=lo,
                max=hi,
                step=(hi - lo) / 200 or 0.01,
                description=c,
                continuous_update=False,
                readout_format=".2f",
                layout=W.Layout(width="520px"),
                style={"description_width": "90px"},
            )
    query = W.Text(
        placeholder="extra pandas query, e.g. op_lead_prob > 0.5",
        description="where",
        continuous_update=False,
        layout=W.Layout(width="720px"),
    )
    button = W.Button(description="show 3 matches")
    out, shown = W.HTML(), W.HTML()

    def _where():
        # a slider applies only on a side that was moved off its end
        parts = []
        for c, s in sliders.items():
            lo, hi = s.value
            if lo > s.min:
                parts.append(f"{c} >= {lo:.4g}")
            if hi < s.max:
                parts.append(f"{c} <= {hi:.4g}")
        if query.value.strip():
            parts.append(f"({query.value.strip()})")
        return " and ".join(parts)

    def summary():
        c, w = conds(), _where()
        print(f"COND = {c!r}\nWHERE = {w!r}")
        profile(where=w or None, df=df, **c)

    def three():
        for sid in _picks(select(_where() or None, df, **conds()), 3):
            show(sid, df=df)

    def refresh(*_):
        out.value = _as_html(summary)

    def show_some(_):
        shown.value = _as_html(three)

    for w in (*pickers.values(), *sliders.values(), query):
        w.observe(refresh, "value")
    button.on_click(show_some)
    refresh()
    display(W.VBox([tab, *sliders.values(), query, button, out, shown]))


explore()

## 12 Failure modes and strengths

`modes(where=, intent=False, **conds)`: for every failure mode, how many keyframes and segments it covers, its median severity, and the scenario, perception or context tag value it concentrates in most (the highest lift among values covering at least a tenth of its keyframes, leaving out the columns its own rule reads). The intent modes (stops, lane changes and turns within the intent horizon that the plan did not anticipate) are listed apart with `intent=True`: the model is never told the route, so they describe information it did not have.

`strengths()` returns the tags file's named strengths, and the scenario clusters (as §9) with at least `min_segments` segments where the plan beats the overall median FDE at H without matching speed or curvature less often than overall.

In [ ]:
MODE_COLUMNS = ["mode", "group", "n", "pct", "segments", "median_severity", "concentrates_in", "doc"]


def modes(where=None, df=None, intent=False, tags=None, **conds):
    """Every failure mode: n, share, segments, median severity, where it concentrates, and its doc."""
    tags = tags or CFG
    sel = select(where, df, **conds)
    rows = []
    for name, spec in tags["failure_modes"].items():
        col = f"fm_{name}"
        if col not in sel or (spec.get("group") == "intent") != intent:
            continue
        hit = sel[sel[col]]
        top = (
            lift(hit, sel, exclude=_names(spec["when"]), top=1, min_n=max(3, len(hit) // 10))
            if len(hit)
            else pd.DataFrame()
        )
        rows.append(
            {
                "mode": name,
                "group": spec.get("group", ""),
                "n": len(hit),
                "pct": 100 * len(hit) / max(len(sel), 1),
                "segments": hit.scene.nunique(),
                "median_severity": hit[f"sev_{name}"].median() if len(hit) else np.nan,
                "concentrates_in": ""
                if top.empty
                else f"{top.tag.iat[0]}={top.value.iat[0]} (x{top.lift.iat[0]:.1f})",
                "doc": spec["doc"],
            }
        )
    return (
        pd.DataFrame(rows, columns=MODE_COLUMNS).set_index("mode").sort_values("n", ascending=False).round(2)
    )


def strengths(
    by=("gt_lat", "gt_long", "speed_band"), min_n=10, min_segments=3, where=None, df=None, tags=None, **conds
):
    """(named, found): the tags file's named strengths, and clusters where the plan does better than overall."""
    tags = tags or CFG
    sel = select(where, df, **conds)
    named = (
        pd.DataFrame(
            [
                {
                    "strength": name,
                    "n": int(sel[f"st_{name}"].sum()),
                    "pct": 100 * sel[f"st_{name}"].mean(),
                    "segments": sel.loc[sel[f"st_{name}"], "scene"].nunique(),
                    "doc": spec["doc"],
                }
                for name, spec in tags["strengths"].items()
                if f"st_{name}" in sel
            ],
            columns=["strength", "n", "pct", "segments", "doc"],
        )
        .set_index("strength")
        .round(2)
    )
    c = clusters(by, min_n, where=where, df=df, **conds)
    speed0, curv0 = 100 * _matched(sel.speed_ratio_bin).mean(), 100 * _matched(sel.curv_ratio_bin).mean()
    ok = (c.segments >= min_segments) & (c.fde_H_median < sel.fde_H.median())
    ok &= ~(c.speed_matched_pct < speed0) & ~(c.curv_matched_pct < curv0)  # NaN (undefined) passes
    return named, c[ok]


with pd.option_context("display.max_colwidth", 80):
    display(modes())
    display(modes(intent=True))
named, found = strengths()
display(named)
print(
    f"clusters that beat the overall median FDE@H ({T.fde_H.median():.2f} m) without matching speed or curvature less often:"
)
display(found)

## 13 Keyframes

`show(key)` draws one keyframe; `key` is a sample id or a row label of `T`. Four panels:

1. **Camera.** The recorded 10 s path (blue ribbon) and the radar tracks, from paddock's comma2k19 overlay, plus the calibrated GT (thin blue line) and the plan (model colour), both projected through the calibration the adapter used. The thin GT line should sit on the ribbon; if it does not, the projection is wrong. paddock's own `render` draws each plan through the same calibration, the record's `rpy_calib_deg`.
2. **BEV** over 10 s: the GT in `GT_FRAME` (the recorded GT dotted when that is `calibrated`), the plan, the points at H circled, the model's lead (square, if its probability is at least `lead_prob`) and the radar's lead (x; labelled clutter when the `lead` tag says so).
3. **Speed** of GT and plan per 0.5 s step, with CAN speed and the model's pose-head speed at t = 0.
4. **Tags**: every class, the ratios and errors, the context, and the failure modes and strengths that fired.

`sheet(mode, n_typical=2, n_worst=1, n_random=1)` draws a failure mode's (or strength's) most severe keyframe, the most typical ones (nearest its median severity) and random ones, at most one per segment. `control()` draws random keyframes whatever their tags, to see what normal looks like. `browse()` steps through a mode's keyframes, most severe first, with `WIDGETS = True` (§1). Each segment is decoded once; its keyframes are cached as JPEG (§0, Outputs).

In [ ]:
FRAMES = cfg.runs / BENCH / "_analysis" / "_frames"
_SEGMENTS, _OVERLAYS = {}, {}


def _segment(r):
    if r.scene not in _SEGMENTS:
        seg = L.find_segment(cfg.datasets[BENCH], int(r.chunk), r.route, int(r.segment))
        if seg is None:
            raise FileNotFoundError(f"segment {r.scene} is under none of this site's {BENCH} roots")
        _SEGMENTS[r.scene] = seg
    return _SEGMENTS[r.scene]


def _frame_path(r, k):
    return FRAMES / f"{int(r.chunk)}_{r.route.replace('|', '_')}_{int(r.segment)}_{int(k)}.jpg"


def frame(i, df=None):
    """The camera frame at row i: its segment is decoded once and every loaded keyframe of it is cached."""
    r = (T if df is None else df).loc[i]
    path = _frame_path(r, r.keyframe)
    if not path.exists():
        want = set(D.loc[D.scene == r.scene, "keyframe"].astype(int))
        FRAMES.mkdir(parents=True, exist_ok=True)
        for k, rgb in enumerate(L.decode_frames(_segment(r).video, limit=max(want) + 1)):
            if k in want:
                Image.fromarray(rgb).save(_frame_path(r, k), quality=92)
    return np.asarray(Image.open(path))


def camera_panel(i, df=None):
    """The frame with the recorded path, radar, the calibrated GT and the plan, through the calibration used."""
    r = (T if df is None else df).loc[i]
    rpy = np.radians(RPY[i])
    if r.scene not in _OVERLAYS:
        layers = OV.Layers(hud=False, minimap=False)
        _OVERLAYS[r.scene] = OV.SegmentOverlay(
            _segment(r).path, layers=layers, path_horizon_s=P["intent_horizon_s"], rpy=rpy
        )
    img = Image.fromarray(_OVERLAYS[r.scene].draw(frame(i, df), int(r.keyframe)))
    pen = ImageDraw.Draw(img)
    gt = rot(GT_RAW[[i]], YAW_CAL[[i]])[0]
    OV.draw_plan(
        pen,
        OV.calib_to_device(OV.ego_to_road(gt[np.isfinite(gt[:, 0])]), rpy),
        palette.GROUND_TRUTH,
        width=2,
        dot=2,
    )
    OV.draw_plan(pen, OV.calib_to_device(OV.ego_to_road(PRED[i]), rpy), palette.color(MODEL))
    return img


def _tag_text(r, h, gt_frame, df):
    groups = tag_columns(df)
    lines = [
        f"{g}: " + ", ".join(f"{c}={r[c]}" for c in cols)
        for g, cols in groups.items()
        if g in ("scenario", "intent", "perception", "response")
    ]
    lines.append(
        f"speed_ratio {r.speed_ratio:.2f}  curv_ratio {r.curv_ratio:.2f}  accel GT {r.gt_accel:+.2f} plan {r.pred_accel:+.2f} m/s²  "
        f"dpsi GT {r.gt_dpsi_deg:+.1f} plan {r.pred_dpsi_deg:+.1f} deg"
    )
    lines.append(
        f"FDE@{h:g}s {r.fde_H:.2f} m ({gt_frame} GT; recorded FDE@3s {r.rec_fde_3s:.2f})  cross {r.cross_H:+.2f} m  z {r.z_H:.1f}"
    )
    if r.pred_comfort_broken_H:
        lines.append(
            f"plan breaks over H: {r.pred_comfort_broken_H} (worst peak {r.pred_comfort_excess_H:.1f}x its bound)"
        )
    ctx = [
        f"{c[4:]}={r[c]}"
        for c in groups.get("context", [])
        if pd.notna(r[c]) and r[c] is not False and r[c] != 0
    ]
    if ctx:
        lines.append("context: " + ", ".join(ctx))
    lines.append(
        "FAILURE MODES: " + (", ".join(c[3:] for c in r.index if c.startswith("fm_") and r[c]) or "none")
    )
    lines.append(
        "strengths: " + (", ".join(c[3:] for c in r.index if c.startswith("st_") and r[c]) or "none")
    )
    return "\n".join(textwrap.fill(line, 64, subsequent_indent="    ") for line in lines)


def _show_photo(fig, quality=80):
    """Display a figure as JPEG. Its camera panel is a photo, which PNG stores several times larger,
    and these figures are most of what an executed copy weighs."""
    buf = io.BytesIO()
    fig.savefig(buf, format="jpeg", bbox_inches="tight", pil_kwargs={"quality": quality})
    plt.close(fig)
    display(
        {"image/jpeg": base64.b64encode(buf.getvalue()).decode(), "text/plain": "<keyframe figure>"}, raw=True
    )


def show(key, params=None, gt_frame=None, camera=True, df=None, title=None):
    """One keyframe: camera, BEV, speed profile and its tags. key: a sample id or a row label of T."""
    df = T if df is None else df
    p, gt_frame = params or P, gt_frame or GT_FRAME
    i = _row(key, df)
    r = df.loc[i]
    h = p["horizon_s"]
    nH = round(h / DT)
    gt = rot(GT_RAW[[i]], YAW_CAL[[i]] if gt_frame == "calibrated" else np.zeros(1))[0]
    fig = plt.figure(figsize=(19, 7.8))
    grid = fig.add_gridspec(2, 3, width_ratios=[2.3, 1, 1.5], height_ratios=[1, 1.25])
    ax_cam, ax_bev = fig.add_subplot(grid[:, 0]), fig.add_subplot(grid[:, 1])
    ax_v, ax_txt = fig.add_subplot(grid[0, 2]), fig.add_subplot(grid[1, 2])

    ax_cam.axis("off")
    if camera:
        try:
            ax_cam.imshow(camera_panel(i, df))
        except Exception as e:  # no dataset or decoder on this machine: say so and draw the rest
            ax_cam.text(
                0.5,
                0.5,
                textwrap.fill(f"no camera: {type(e).__name__}: {e}", 60),
                ha="center",
                va="center",
                transform=ax_cam.transAxes,
            )

    if gt_frame == "calibrated":
        ax_bev.plot(
            GT_RAW[i, :, 1], GT_RAW[i, :, 0], ":", color=palette.GROUND_TRUTH, lw=1.2, label="GT as recorded"
        )
    bev.draw(ax_bev, {MODEL: PRED[i]}, ground_truth=gt[np.isfinite(gt[:, 0])])
    ax_bev.plot(gt[nH - 1, 1], gt[nH - 1, 0], "o", mfc="none", color=palette.GROUND_TRUTH, ms=10)
    ax_bev.plot(
        PRED[i, nH - 1, 1],
        PRED[i, nH - 1, 0],
        "o",
        mfc="none",
        color=palette.color(MODEL),
        ms=10,
        label=f"at H = {h:g} s",
    )
    if r.op_lead_prob >= p["lead_prob"]:
        ax_bev.plot(
            r.op_lead_y,
            r.op_lead_d,
            "s",
            mfc="none",
            color=palette.color(MODEL),
            ms=9,
            label=f"model lead (p {r.op_lead_prob:.2f})",
        )
    if np.isfinite(r.ref_lead_d):
        ax_bev.plot(
            r.ref_lead_y,
            r.ref_lead_d,
            "kx",
            ms=9,
            label="radar lead (clutter)" if r.lead == "clutter" else "radar lead",
        )
    lim = max(3.0, 1.2 * np.nanmax(np.abs(np.r_[gt[:, 1], PRED[i, :, 1]])))
    ax_bev.set_aspect("auto")
    ax_bev.set_xlim(lim, -lim)  # left positive on the left, as bev.draw
    ax_bev.legend(fontsize=7, loc="upper left")
    ax_bev.set_title(f"BEV, GT {gt_frame}", fontsize=9)

    t_mid = (np.arange(N_WP) + 0.5) * DT
    for xy, color, label in ((gt, palette.GROUND_TRUTH, "GT"), (PRED[i], palette.color(MODEL), MODEL)):
        v = np.linalg.norm(np.diff(np.vstack([[0.0, 0.0], xy]), axis=0), axis=1) / DT
        ax_v.plot(t_mid, v, ".-", color=color, label=label)
    ax_v.plot(0, r.ref_speed, "k>", ms=8, label="CAN speed")
    ax_v.plot(0, r.op_v, "<", color=palette.color(MODEL), ms=8, label="pose-head speed")
    ax_v.axvline(h, color="grey", ls="--", lw=0.8)
    ax_v.set(xlabel="t (s)", ylabel="speed (m/s)")
    ax_v.grid(alpha=0.3)
    ax_v.legend(fontsize=7)

    ax_txt.axis("off")
    ax_txt.text(
        0,
        1,
        _tag_text(r, h, gt_frame, df),
        va="top",
        family="monospace",
        fontsize=7.5,
        transform=ax_txt.transAxes,
    )
    fig.suptitle(
        f"{title + ': ' if title else ''}{r.sample_id}   {r.vehicle}, {r.ref_speed:.1f} m/s", fontsize=11
    )
    fig.tight_layout()
    _show_photo(fig)


def sheet(mode, n_typical=2, n_worst=1, n_random=1, seed=None, df=None, **kw):
    """A failure mode's (or strength's) worst, most typical and random keyframes, at most one per segment."""
    df = T if df is None else df
    col = mode if mode.startswith(("fm_", "st_")) else f"fm_{mode}" if f"fm_{mode}" in df else f"st_{mode}"
    sev = f"sev_{col[3:]}"
    hit = df[df[col]]
    print(f"{col}: {len(hit)} keyframes over {hit.scene.nunique()} segments")
    if hit.empty:
        return []
    ranked = sev in hit
    if ranked:  # each segment's most severe keyframe stands for it
        hit = hit.sort_values(sev, ascending=False)
    hit = hit.drop_duplicates("scene")
    picks = [("worst", i) for i in hit.index[:n_worst]] if ranked else []
    if ranked and n_typical:
        rest = hit.drop([i for _, i in picks])
        picks += [
            ("typical", i) for i in (rest[sev] - rest[sev].median()).abs().sort_values().index[:n_typical]
        ]
    rest = hit.drop([i for _, i in picks])
    n_more = n_random + (0 if ranked else n_worst + n_typical)
    if n_more and len(rest):
        picks += [
            ("random", i)
            for i in rest.sample(min(n_more, len(rest)), random_state=SEED if seed is None else seed).index
        ]
    for why, i in picks[:MAX_RENDER]:
        if RENDER:
            show(
                i,
                df=df,
                title=f"{col[3:]}, {why}" + (f" (severity {df.at[i, sev]:.2f})" if ranked else ""),
                **kw,
            )
        else:
            print(f"  {why}: {df.at[i, 'sample_id']}")
    return [df.at[i, "sample_id"] for _, i in picks[:MAX_RENDER]]


def control(n=8, seed=None, df=None):
    """Random keyframes, one per segment, whatever their tags: what normal looks like."""
    df = T if df is None else df
    for sid in _picks(df, min(n, MAX_RENDER), seed):
        show(sid, df=df, title="control")


def browse(df=None):
    """A failure mode (or strength) dropdown and a rank slider: its keyframes, most severe first."""
    if _no_widgets("call sheet(...) or show(...)"):
        return
    df = T if df is None else df
    cols = [c for c in df.columns if c.startswith(("fm_", "st_")) and df[c].any()]
    if not cols:
        print("no failure mode or strength fired on the loaded keyframes")
        return
    pick = W.Dropdown(options=[(f"{c} ({int(df[c].sum())})", c) for c in cols], description="mode")
    rank = W.IntSlider(value=0, min=0, max=0, description="rank")
    out = W.HTML()

    def ranked():
        hit = df[df[pick.value]]
        sev = f"sev_{pick.value[3:]}"
        return hit.sort_values(sev, ascending=False) if sev in hit else hit

    def draw(*_):
        hit = ranked()
        k = min(rank.value, len(hit) - 1)
        out.value = _as_html(show, hit.index[k], df=df, title=f"{pick.value[3:]} #{k + 1} of {len(hit)}")

    def on_mode(*_):  # a new mode starts at rank 0 and draws once, not once per slider change
        rank.unobserve(draw, "value")
        rank.max, rank.value = len(ranked()) - 1, 0
        rank.observe(draw, "value")
        draw()

    rank.observe(draw, "value")
    pick.observe(on_mode, "value")
    on_mode()
    display(W.VBox([W.HBox([pick, rank]), out]))

In [ ]:
if FOCUS:  # debugging: every loaded keyframe
    for i in T.index[:MAX_RENDER]:
        show(i)
else:  # the most frequent failure modes: the worst and the most typical keyframe of each
    for name in modes().query("n > 0").index[:6]:
        sheet(name, n_typical=1, n_worst=1, n_random=0)

In [ ]:
control(4)

In [ ]:
browse()

## 14 Export

`export()` writes the tables behind this notebook, for the loaded keyframes and §1's parameters, to `{runs}/comma2k19/_analysis/openpilot/<run_id>/<subset>/`: `tags.csv` (all of `T`), `clusters.csv`, `modes.csv`, `modes_intent.csv`, `strengths_named.csv`, `strengths_found.csv`, `scores.csv` (both GT frames), `glossary.md`, and `manifest.json` (the run, `sequences_sha256`, subset or focus, `GT_FRAME`, `OVERRIDES`, and hashes of the tags config and params). `<subset>` is `pilot`, `all`, `random-<fraction>-s<seed>` or `focus-<hash>`. Nothing is written under `results/`.

In [ ]:
def export(df=None, slug=None):
    """Write the tables and a manifest under {runs}/<bench>/_analysis/<model>/<run_id>/<slug>/."""
    df = T if df is None else df
    if slug is None:
        if FOCUS:
            slug = "focus-" + hashlib.sha1(json.dumps(FOCUS).encode()).hexdigest()[:8]
        elif SUBSET == "random":
            slug = f"random-{FRACTION:g}-s{SEED}"
        else:
            slug = SUBSET
    out = cfg.runs / BENCH / "_analysis" / MODEL / RUN.run_id / slug
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "tags.csv", index=False)
    clusters(df=df).to_csv(out / "clusters.csv")
    modes(df=df).to_csv(out / "modes.csv")
    modes(df=df, intent=True).to_csv(out / "modes_intent.csv")
    named, found = strengths(df=df)
    named.to_csv(out / "strengths_named.csv")
    found.to_csv(out / "strengths_found.csv")
    by_frame(df=df).to_csv(out / "scores.csv")
    (out / "glossary.md").write_text(f"{glossary()}\n\n### Feature columns\n\n{feature_glossary()}\n")
    manifest = {
        "run_id": RUN.run_id,
        "benchmark": BENCH,
        "model": MODEL,
        "sequences_sha256": RUN.run.get("dataset", {}).get("sequences_sha256"),
        "subset": SUBSET,
        "focus": FOCUS,
        "fraction": FRACTION,
        "seed": SEED,
        "n_keyframes": len(df),
        "gt_frame": GT_FRAME,
        "bench_params": BENCH_PARAMS,
        "overrides": OVERRIDES,
        "tags_file": TAGS.relative_to(REPO_ROOT).as_posix(),
        "tags_hash": config_hash(CFG),
        "params_hash": config_hash(P),
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
    print(
        f"wrote {{runs}}/{out.relative_to(cfg.runs).as_posix()}/: "
        + ", ".join(sorted(f.name for f in out.iterdir()))
    )


export()

## 15 Scratch

Everything above is a function of keyword arguments, so experiments go here. Some starting points:

```python
T2 = retag(params=P | {"a_brake": -0.7, "horizon_s": 2.0}); confusion("long", df=T2)   # another threshold, side by side
scores(by="lead_detect", horizons=(1, 3, 5), stat="mean")                              # errors by perception outcome
profile(fm_phantom_brake=True)                                                         # what phantom braking looks like
heatmap("gt_long", "speed_band", value="fm_under_brakes")                              # where under-braking happens
dist(["gt_accel", "pred_accel", "accel_delta"], by="speed_band")                       # tune the braking thresholds
explain(select(fm_confidently_wrong=True).sample_id.iloc[0])                           # why one keyframe was tagged
sheet("understeer", n_typical=3, n_worst=2, n_random=2)                                # more keyframes of one mode
scores(by="vehicle", gt_frame="recorded", yaw_offset_deg=1.5)                          # re-check the frame finding by hand
```